# CTD Profile Grapher

Interactive depth profiles from Sea-Bird `.cnv` files — no Excel step, no Google Drive.

**How to use**
1. Run **Setup** once per session.
2. Run **1 · Survey location and files** — type where you sampled, pick your `.cnv` files, then tick which variables you want graphed. The ones the notebook recognises are already ticked; anything else your files contain is listed too, in case you want it. Depth is the Y axis unless you change it.
3. Run **2 · Draw the graphs**. To look at part of the water column, fill in `depth_from_m` and `depth_to_m` and run that cell again. Your files stay loaded, so there is no need to upload twice.
4. For a station map or a transect, run **3 · Station positions and transect** — positions are filled in where the file knows them; type in the rest, tick the stations on the line, drag them into order, add any seafloor depths you know between stations — then **4 · Station map and water column transect**. Skip both if you only want the graphs.

**Naming:** the filename becomes the legend label, with underscores turned into spaces — `Station_1.cnv` → **Station 1**, `East_Passage.cnv` → **East Passage**. Stations sort naturally (1, 2, … 10, 11).

**Colours** are locked to station order, so the first station is always the same blue, the second always the same red, and so on. The Load cell prints the colour key so a bar chart or a station map can use exactly the same colours.

## What you get

Files land in `/content/CTD_output/` — open the folder icon in the left sidebar to download them.

- **`png/*.png`** — ordinary pictures, one per variable. Use these anywhere you would use a photo.
- **`CTD_profiles.html`** — all the graphs together, interactive, around 30 KB. Hovering shows exact values; you can zoom, or hide a station by clicking it in the legend. Needs internet to open.
- **`single_graphs/*.html`** — the same graphs but **one file each**: `Temperature.html`, `Salinity.html`, and so on. Take just the one you want.
- **`CTD_profiles_offline.html`** — all the graphs with everything built in, several MB, works with no connection at all. Only worth taking if you will be presenting somewhere without wifi.

## Sharing a graph

**For a document or slideshow — use the PNG.** Google Docs, Word, Google Slides and PowerPoint cannot display an interactive chart, and no setting or add-on changes that. Insert the picture like any other image.

**To let someone explore it — send them `CTD_profiles.html`.** They double-click it and it opens in their browser with everything working, nothing to install. This is the easy answer and it covers almost every case.

**Want just one graph, not all of them?** Use the matching file from `single_graphs/` — `Temperature.html` is the depth vs temperature graph on its own, and nothing else. Treat it exactly like the file above: send it, or put it online for a link. Each one is about 15 KB.

**To put a graph inside a web page**, hand the HTML file to whoever manages the site and ask them to embed it in an iframe. That is a normal request and they will know what it means. Google Drive will not work as a substitute — it downloads the file rather than displaying it.

The sensor set is read from each file's own header, so casts from different instruments work with no setting to change.

## Credit

Example data collected by students of **TGEOS 445, Estuarine Field Studies, University of Washington Tacoma**, Spring 2026, in Colvos Passage and East Passage, Puget Sound.

Instrument: **Sea-Bird SBE 19plus** (temperature and conductivity SN 7686), processed with Sea-Bird SBEDataProcessing.

In [ ]:
#@title Setup — run once per session
!pip install -q "kaleido==0.2.1" 2>/dev/null

import os, re, colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ─────────────── settings ───────────────
LINE_SHAPE   = "spline"              # "spline" (smooth) or "linear" (raw bins)
SHOW_MARKERS = False                 # True → dot at each 1-db bin
# How the profile graphs look. The Draw cell sets these four from its form.
LEGEND_POS   = "right"               # "left", "right" or "bottom"
DARK_GRAPHS  = False                 # True → dark background, light ink
SHOW_TITLES  = True                  # False → no title above each graph
Y_LABEL_TOP  = False                 # True → y label upright above the axis
LINE_WIDTH   = 1.5
X_PAD_FRAC   = 0.05                  # breathing room at left/right edges (5%)
Y_PAD_FRAC   = 0.02
AXIS_FONT    = 13                    # x and y axis titles share this size
EXPORT_PNG   = True
OUT_DIR      = "/content/CTD_output"
LOCATION     = ""                    # set by the Load cell, e.g. "Quartermaster Harbor"
# ────────────────────────────────────────

# (label, ticked by default, [(short name, unit), ...])
# The classic set is ticked; derived and engineering channels are recognised so
# they get a proper name and unit, but stay unticked so the default output does
# not balloon. Names surveyed across 51 public .cnv files from SBE 9, 19plus,
# 21 and 25plus instruments.
VARIABLES = [
    ("Temperature", True,
     [("tv290c", "°C"), ("t090c", "°C"), ("t190c", "°C"), ("tv190c", "°C"),
      ("t068c", "°C"), ("t168c", "°C"), ("t090", "°C"), ("t190", "°C"),
      ("t4990c", "°C"), ("tnc90c", "°C")]),
    ("Potential Temperature", False,
     [("potemp090c", "°C"), ("potemp190c", "°C"), ("potemp068c", "°C"),
      ("potemp168c", "°C")]),
    ("Salinity", True, [("sal00", "PSU"), ("sal11", "PSU")]),
    ("Conductivity", False,
     [("c0s/m", "S/m"), ("c1s/m", "S/m"), ("c0ms/cm", "mS/cm"),
      ("c1ms/cm", "mS/cm"), ("c0us/cm", "µS/cm"), ("c1us/cm", "µS/cm"),
      ("cond0s/m", "S/m")]),
    ("Density (sigma-t)", True,
     [("sigma-t00", "kg/m³"), ("sigma-t11", "kg/m³"), ("sigma-e00", "kg/m³"),
      ("sigma-é00", "kg/m³"), ("sigma-é11", "kg/m³"), ("density00", "kg/m³"),
      ("density11", "kg/m³")]),
    ("Sound Velocity", False,
     [("svcm", "m/s"), ("avgsvcm", "m/s"), ("svcm1", "m/s")]),
    ("Dissolved Oxygen", True,
     [("sbeox0ml/l", "mL/L"), ("sbeox1ml/l", "mL/L"), ("oxml/l", "mL/L"),
      ("sbeox0mm/kg", "µmol/kg"), ("sbeox1mm/kg", "µmol/kg"),
      ("sbeox0mg/l", "mg/L"), ("sbeox1mg/l", "mg/L"),
      ("sbeox0ps", "% sat"), ("sbeox1ps", "% sat")]),
    ("Oxygen Solubility", False,
     [("oxsolmm/kg", "µmol/kg"), ("oxsolml/l", "mL/L"), ("oxsolmg/l", "mg/L")]),
    ("Oxygen Saturation", False,
     [("oxsatmm/kg", "µmol/kg"), ("oxsatml/l", "mL/L"), ("oxsatmg/l", "mg/L")]),
    ("Fluorescence", True,
     [("fleco-afl", "mg/m³"), ("flecoafl", "mg/m³"), ("flcuva", "mg/m³"),
      ("flsp", "mg/m³"), ("flc", "mg/m³"), ("wetstar", "mg/m³")]),
    ("Beam Transmission", True,
     [("cstartr0", "%"), ("cstartr1", "%"), ("xmiss", "%")]),
    ("Beam Attenuation", False,
     [("bat", "1/m"), ("cstarat0", "1/m"), ("cstarat1", "1/m")]),
    ("Turbidity", True,
     [("turbwetntu0", "NTU"), ("turbwetntu1", "NTU"), ("obs", "NTU"),
      ("obs3+", "NTU"), ("seaturbmtr", "NTU"), ("upoly0", "NTU")]),
    ("pH", True, [("ph", "")]),
    ("PAR", True, [("par", "µmol photons/m²/s"), ("spar", "µmol photons/m²/s")]),
    ("CDOM", True, [("wetcdom", "mg/m³")]),
    ("Specific Volume Anomaly", False, [("sva", "10⁻⁸ m³/kg")]),
    ("Thermosteric Anomaly", False, [("tsa", "10⁻⁸ m³/kg")]),
]

# Vertical axis. Depth is preferred; pressure stands in when a file has no depth
# channel, and is labelled as pressure rather than quietly called metres.
DEPTH_CANDIDATES = [("depsm", "Depth (m)"), ("depfm", "Depth (m)"),
                    ("depsf", "Depth (fathoms)"),
                    ("prdm", "Pressure (db)"), ("prsm", "Pressure (db)"),
                    ("prm", "Pressure (db)"), ("prde", "Pressure (db)"),
                    ("pr", "Pressure (db)"), ("prdb", "Pressure (db)")]

# Fixed station palette. Position decides colour: the 1st station in the
# canonical order is always PALETTE[0], the 2nd always PALETTE[1], and so on.
# Reuse STATION_COLORS in a bar chart or a station map and the colours agree
# across every figure you make.
PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd",
           "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]

STATION_COLORS = {}
STATION_META = {}          # station label -> parse_header_meta() result
_EXTRA_COLORS = []          # generated colours beyond PALETTE, in order
_LAB = {}


def _to_lab(hexcolor):
    """sRGB hex → CIE Lab, so colours can be compared the way an eye does.
    Plain RGB distance calls greens near-identical that clearly are not."""
    if hexcolor in _LAB:
        return _LAB[hexcolor]
    r, g, b = (int(hexcolor[i:i + 2], 16) / 255 for i in (1, 3, 5))
    inv = lambda c: c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = inv(r), inv(g), inv(b)
    x = (0.4124 * r + 0.3576 * g + 0.1805 * b) / 0.95047
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    z = (0.0193 * r + 0.1192 * g + 0.9505 * b) / 1.08883
    f = lambda t: t ** (1 / 3) if t > 0.008856 else 7.787 * t + 16 / 116
    x, y, z = f(x), f(y), f(z)
    _LAB[hexcolor] = (116 * y - 16, 500 * (x - y), 200 * (y - z))
    return _LAB[hexcolor]


def _candidate_colors():
    out = []
    for h in range(36):
        for light in (0.36, 0.50, 0.64):
            for sat in (0.50, 0.75, 0.95):
                r, g, b = colorsys.hls_to_rgb(h / 36, light, sat)
                out.append("#{:02x}{:02x}{:02x}".format(
                    round(r * 255), round(g * 255), round(b * 255)))
    return out


def _ensure_colors(n):
    """Grow the colour list to n entries, each new colour chosen as the one
    furthest from every colour already in use. Greedy farthest-point picking
    keeps large sets readable where evenly-spaced hues do not."""
    if len(PALETTE) + len(_EXTRA_COLORS) >= n:
        return
    cands = _candidate_colors()
    used = [_to_lab(c) for c in PALETTE + _EXTRA_COLORS]
    while len(PALETTE) + len(_EXTRA_COLORS) < n:
        best, best_d = None, -1.0
        for c in cands:
            lc = _to_lab(c)
            d = min((lc[0] - u[0]) ** 2 + (lc[1] - u[1]) ** 2 + (lc[2] - u[2]) ** 2
                    for u in used)
            if d > best_d:
                best_d, best = d, c
        _EXTRA_COLORS.append(best)
        used.append(_to_lab(best))


def station_color(i):
    """Colour for the i-th station. Lines are always solid, so past the base
    palette new colours are generated rather than repeated. Deterministic:
    station i gets the same colour every run, for any n."""
    if i < len(PALETTE):
        return PALETTE[i]
    _ensure_colors(i + 1)
    return _EXTRA_COLORS[i - len(PALETTE)]


def assign_station_styles(labels):
    """Lock each station to a colour by its position in the canonical order.

    Call once after loading. The returned dict is the colour key for this
    survey — reuse it in any other chart of the same stations."""
    STATION_COLORS.clear()
    for i, lab in enumerate(labels):
        STATION_COLORS[lab] = station_color(i)
    return STATION_COLORS


def natkey(s):
    """Sort so Station 2 comes before Station 10."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


# Colab's uploader never overwrites: upload Station_1.cnv twice and the second
# copy lands as "Station_1 (1).cnv", counting up on each repeat.
DUP_SUFFIX = re.compile(r"\s*\((\d+)\)\s*$")


def station_label(filename):
    """Station_1.cnv -> 'Station 1';  East_Passage.cnv -> 'East Passage'.
    A trailing ' (1)' left by a repeated upload is dropped."""
    base = os.path.splitext(os.path.basename(filename))[0]
    return DUP_SUFFIX.sub("", base).replace("_", " ").strip()


def upload_generation(filename):
    """How many times this file has been re-uploaded. Colab counts upward, so
    the highest number is the most recent copy."""
    base = os.path.splitext(os.path.basename(filename))[0]
    m = DUP_SUFFIX.search(base)
    return int(m.group(1)) if m else 0


def dedupe_uploads(raw):
    """Keep only the newest copy of each station.

    Re-uploading to fix a mistake would otherwise plot the station twice — once
    from the bad file and once from the good one — and shift every station's
    colour, since colour follows position."""
    best = {}
    for fn in raw:
        lab, gen = station_label(fn), upload_generation(fn)
        if lab not in best or gen > best[lab][0]:
            best[lab] = (gen, fn)
    keep = {fn for _, fn in best.values()}
    for fn in raw:
        if fn not in keep:
            print(f"  IGNORED  {fn} — superseded by a newer upload of "
                  f"'{station_label(fn)}'")
    return {fn: raw[fn] for fn in raw if fn in keep}


def find_channel(df, pairs):
    """First matching (short name, unit) pair, case-insensitive.
    Returns (column, unit), or (None, None) if the file carries none of them.
    First occurrence wins when a name appears twice."""
    lower = {}
    for c in df.columns:
        lower.setdefault(str(c).lower(), c)
    for name, unit in pairs:
        if name in lower:
            return lower[name], unit
    return None, None


def downcast_only(df, depth_col):
    """Keep the downcast only: cut at the deepest reading, then keep readings
    deeper than everything before them. Returns (cleaned, rows_removed,
    time_series).

    A record that is not a cast at all — a moored instrument logging at one
    depth for weeks — would be gutted by that cut, so when it would keep under
    a fifth of the rows the frame is left alone and time_series is True.

    This makes a profile readable. It is NOT Sea-Bird's processing and is no
    substitute for it — see the Raw casts section of the README, and
    https://github.com/HakaiInstitute/seabird-processing"""
    if depth_col is None or len(df) < 3:
        return df, 0, False
    d = pd.to_numeric(df[depth_col], errors="coerce").to_numpy(dtype=float)
    if np.all(np.isnan(d)):
        return df, 0, False
    down = df.iloc[:int(np.nanargmax(d)) + 1]
    dd = pd.to_numeric(down[depth_col], errors="coerce").to_numpy(dtype=float)
    running = np.maximum.accumulate(np.nan_to_num(dd, nan=-np.inf))
    cleaned = down[dd >= running]
    if len(df) >= 50 and len(cleaned) < len(df) * 0.2:
        return df, 0, True
    return cleaned, len(df) - len(cleaned), False


# Instrument bookkeeping rather than measurements — never offered as plottable.
HOUSEKEEPING = {"scan", "flag", "nbin", "pumps", "bpos", "pla", "timej", "times",
                "timem", "timek", "timen", "latitude", "longitude", "dz/dtm", "accm",
                "altm", "nbf", "prfails", "moderror", "seconds"}


def _nmea_degrees(text):
    """'25 55.56 S' -> -25.926.  Sea-Bird writes degrees, decimal minutes and a
    hemisphere; south and west are negative."""
    m = re.match(r"\s*(\d+)\s+([\d.]+)\s*([NSEW])", text.strip(), re.I)
    if not m:
        return None
    deg = int(m.group(1)) + float(m.group(2)) / 60.0
    return -deg if m.group(3).upper() in ("S", "W") else deg


def parse_header_meta(text):
    """Pull cast metadata out of a .cnv header.

    Headers carry three tiers: '*' instrument config, '**' free-form discovery
    metadata (ship, station, water depth), and '#' processing and parameters.
    Coordinates live in the '*' NMEA lines and are present in most real files,
    so a transect rarely needs them typed in by hand.

    Every field is optional — returns None for anything absent."""
    head = text.split("*END*")[0]
    out = {"lat": None, "lon": None, "time": None, "station": None,
           "ship": None, "cruise": None, "water_depth": None, "instrument": None}

    for line in head.splitlines():
        s = line.strip()
        m = re.match(r"\*\s*NMEA\s+Latitude\s*=\s*(.+)", s, re.I)
        if m:
            out["lat"] = _nmea_degrees(m.group(1))
            continue
        m = re.match(r"\*\s*NMEA\s+Longitude\s*=\s*(.+)", s, re.I)
        if m:
            out["lon"] = _nmea_degrees(m.group(1))
            continue
        # NMEA time is often literally "none"; fall back to start_time.
        m = re.match(r"\*\s*NMEA\s+UTC\s*\(Time\)\s*=\s*(.+)", s, re.I)
        if m and "none" not in m.group(1).lower():
            out["time"] = out["time"] or m.group(1).strip()
            continue
        m = re.match(r"#\s*start_time\s*=\s*([^\[]+)", s, re.I)
        if m:
            out["time"] = out["time"] or m.group(1).strip()
            continue
        if out["instrument"] is None:
            m = re.match(r"\*\s*(Sea-?Bird\s+SBE[^\r\n]*?)\s*Data File", s, re.I)
            if m:
                out["instrument"] = m.group(1).strip()
                continue
        # '**' lines are free-form "Key: value" or "Key = value"; only a handful
        # of files use them, and the keys are not standardised.
        m = re.match(r"\*\*\s*([^:=]+)[:=]\s*(.+)", s)
        if m:
            key, val = m.group(1).strip().lower(), m.group(2).strip()
            if not val:
                continue
            if "stat" in key:
                out["station"] = out["station"] or val
            elif "ship" in key or "vessel" in key:
                out["ship"] = out["ship"] or val
            elif "cruise" in key:
                out["cruise"] = out["cruise"] or val
            elif "depth" in key:
                d = re.search(r"[-+]?\d*\.?\d+", val)
                if d:
                    out["water_depth"] = float(d.group())
    return out


def _col_by_name(df, name):
    """Exact column lookup, case-insensitive."""
    for c in df.columns:
        if str(c).lower() == str(name).lower():
            return c
    return None


def available_channels(stations):
    """What these casts can plot.

    Returns (recognised, extras, y_default). `recognised` are the variables the
    notebook knows by name, with units. `extras` is every other measured column
    the files happen to carry — offered so nothing is hidden, just unticked by
    default. `y_default` is the depth or pressure channel."""
    recognised, seen = [], set()
    for label, on_by_default, cands in VARIABLES:
        for _st, df in stations:
            col, unit = find_channel(df, cands)
            if col is not None:
                recognised.append({"name": label, "col": str(col), "on": on_by_default,
                                   "label": f"{label} ({unit})" if unit else label})
                seen.add(str(col).lower())
                break

    ycol, ylab = None, None
    for _st, df in stations:
        c, l = find_channel(df, DEPTH_CANDIDATES)
        if c is not None:
            ycol, ylab = str(c), l
            break
    if ycol:
        seen.add(ycol.lower())

    extras = []
    for _st, df in stations:
        for c in df.columns:
            lc = str(c).lower()
            if lc in seen or lc in HOUSEKEEPING or (lc.startswith("v") and lc[1:].isdigit()):
                continue
            seen.add(lc)
            extras.append({"name": str(c), "col": str(c), "label": str(c)})

    # is_depth drives the depth window and the surface anchoring; invert only
    # flips the axis. They are separate so unticking "invert" does not quietly
    # disable the depth window as well.
    y_default = {"name": (ylab or "Depth (m)").split(" (")[0],
                 "col": ycol, "label": ylab or "Depth (m)",
                 "is_depth": True, "invert": True}
    return recognised, extras, y_default


def parse_cnv(text, source_name):
    """Parse a Sea-Bird .cnv. Columns come from the '# name' header lines and the
    data starts after *END*, so header length never has to be hardcoded."""
    lines = text.splitlines()
    col_names, bad_flag, end_idx, processing = {}, -9.99e-29, None, set()

    for i, raw in enumerate(lines):
        s = raw.strip()
        if s.upper() == "*END*":
            end_idx = i
            break
        m = re.match(r"#\s*name\s+(\d+)\s*=\s*([^:]+):", s)
        if m:
            col_names[int(m.group(1))] = m.group(2).strip()
            continue
        m = re.match(r"#\s*bad_flag\s*=\s*(\S+)", s)
        if m:
            try:
                bad_flag = float(m.group(1))
            except ValueError:
                pass
            continue
        low = s.lower()
        for tag in ("loopedit", "binavg", "wfilter", "filter", "derive",
                    "alignctd", "celltm", "split", "wildedit"):
            if low.startswith("# " + tag):
                processing.add(tag)

    if end_idx is None:
        raise ValueError(f"{source_name}: no *END* marker — is this a Sea-Bird .cnv?")
    if not col_names:
        raise ValueError(f"{source_name}: no '# name' column definitions in header.")

    ordered = [col_names[k] for k in sorted(col_names)]
    # A name that appears twice (some files carry depSM twice) would make
    # df[name] a two-column frame, so repeats are numbered; the first keeps
    # the plain name and is the one the notebook picks.
    seen = {}
    for i, name in enumerate(ordered):
        n = seen.get(name.lower(), 0)
        seen[name.lower()] = n + 1
        if n:
            ordered[i] = f"{name}.{n}"
    ncol = len(ordered)
    rows = []
    for raw in lines[end_idx + 1:]:
        parts = raw.split()
        # Sea-Bird writes 11-character fields, so a wide negative number can run
        # into its neighbour with no space between (13.5181-218.764914). When
        # the split gives the wrong count, cut the line at fixed width instead.
        if len(parts) != ncol and len(raw) >= ncol * 11 - 1:
            fixed = [raw[c * 11:c * 11 + 11].strip() for c in range(ncol)]
            if all(fixed):
                parts = fixed
        if len(parts) != ncol:
            continue
        try:
            rows.append([float(x) for x in parts])
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=ordered)
    if not df.empty:
        # bad_flag is ~1e-29, so the comparison must be purely relative (atol=0),
        # otherwise every near-zero reading would be wiped out.
        mask = np.isclose(df.values.astype(float), bad_flag, rtol=1e-6, atol=0.0)
        df = df.mask(pd.DataFrame(mask, index=df.index, columns=df.columns))
    return df, processing


def load_files():
    """Show the upload button and read whatever is picked."""
    from google.colab import files as colab_files
    print("Select one or more .cnv files:")
    # .cnv headers are cp1252, not UTF-8 (e.g. the theta in sigma-theta)
    return dedupe_uploads(
        {n: b.decode("latin-1") for n, b in colab_files.upload().items()})


def pretty_units(u, html=True):
    """Units as Sea-Bird writes them, made readable: superscripts, the degree
    sign, micro. html=True gives Plotly markup (<sup>), html=False gives
    Unicode for plain text."""
    s = str(u or "").strip()
    s = re.sub(r"^(ITS-90|ITS-68|IPTS-68),?\s*", "", s, flags=re.I)
    s = re.sub(r"^PSS-78,?\s*", "", s, flags=re.I)
    s = re.sub(r"^sigma-(t|theta|θ|é),\s*", "", s, flags=re.I)
    s = re.sub(r"deg C", "°C", s, flags=re.I)
    s = re.sub(r"\bumol\b", "µmol", s, flags=re.I)
    s = re.sub(r"\bug\b", "µg", s)
    s = re.sub(r"\s+\]$", "", s).strip()
    if html:
        return re.sub(r"\^(-?\d+)", lambda m: "<sup>" + m.group(1) + "</sup>", s)
    sup = {"0": "⁰", "1": "¹", "2": "²", "3": "³", "4": "⁴", "5": "⁵",
           "6": "⁶", "7": "⁷", "8": "⁸", "9": "⁹", "-": "⁻"}
    return re.sub(r"\^(-?\d+)", lambda m: "".join(sup.get(c, c) for c in m.group(1)), s)


def label_with_units(name, units, html=True):
    """'Temperature (°C)' style label, or just the name when there are no units."""
    u = pretty_units(units, html)
    return f"{name} ({u})" if u else name


def _padded(lo, hi, frac):
    """Range with breathing room. Falls back sensibly if the series is flat."""
    span = hi - lo
    pad = span * frac if span > 0 else (abs(hi) * frac if hi else 1.0) or 1.0
    return lo - pad, hi + pad


def _pretty_label(label, html=True):
    """A typed axis label with its units tidied, so 'Temperature (deg C)' reads
    'Temperature (°C)' and mg/m^3 gets a real superscript."""
    m = re.match(r"^(.*?)\s*\(([^()]*)\)\s*$", str(label or "").strip())
    if m:
        return label_with_units(m.group(1), m.group(2), html)
    return pretty_units(label, html)


def _graph_theme():
    """Layout keys for the light or dark look. The dark surface is the web
    app's, so graphs from either look the same side by side."""
    if DARK_GRAPHS:
        return dict(template="plotly_dark", paper_bgcolor="#172028",
                    plot_bgcolor="#172028", font=dict(color="#e6ecef"))
    return dict(template="plotly_white")


def build_figures(stations, series, y_channel, depth_min=None, depth_max=None):
    """One figure per entry in `series`, every station overlaid.

    series      [{"name","label","col"}, ...]  x axis, one figure each
    y_channel   {"name","label","col","invert"}  shared y axis

    y_channel["is_depth"] says whether the depth window and surface anchoring
    apply; y_channel["invert"] only flips the axis direction, and is the user's
    tick box. Put a non-depth variable on y and it becomes an ordinary scatter —
    salinity against temperature, say — where no depth window makes sense.

    LEGEND_POS, DARK_GRAPHS, SHOW_TITLES and Y_LABEL_TOP (Setup settings, set
    from the Draw cell's form) decide how the figures look."""
    if not STATION_COLORS:
        assign_station_styles([s for s, _ in stations])
    is_depth = bool(y_channel.get("is_depth", False))
    invert = bool(y_channel.get("invert", False))
    ylabel = _pretty_label(y_channel["label"])
    figs = []

    for s in series:
        xlabel = _pretty_label(s["label"])
        traces = []
        xlo = ylo = np.inf
        xhi = yhi = -np.inf
        for i, (st, df) in enumerate(stations):
            xcol = _col_by_name(df, s["col"])
            ycol = _col_by_name(df, y_channel["col"])
            if xcol is None or ycol is None:
                continue
            # The depth window filters readings by how deep they were taken, so it
            # applies whatever is on the axes — including salinity against
            # temperature, where depth is not plotted at all.
            dcol, _ = find_channel(df, DEPTH_CANDIDATES)
            cols = list(dict.fromkeys([c for c in (xcol, ycol, dcol) if c is not None]))
            sub = df[cols].dropna(subset=[xcol, ycol])
            if dcol is not None and depth_min is not None:
                sub = sub[sub[dcol] >= depth_min]
            if dcol is not None and depth_max is not None:
                sub = sub[sub[dcol] <= depth_max]
            if sub.empty:
                continue
            xlo, xhi = min(xlo, sub[xcol].min()), max(xhi, sub[xcol].max())
            ylo, yhi = min(ylo, sub[ycol].min()), max(yhi, sub[ycol].max())
            traces.append(go.Scatter(
                x=sub[xcol], y=sub[ycol], name=st,
                mode="lines+markers" if SHOW_MARKERS else "lines",
                line=dict(shape=LINE_SHAPE, width=LINE_WIDTH,
                          color=STATION_COLORS.get(st, station_color(i))),
                marker=dict(size=4),
                hovertemplate=(f"<b>{st}</b><br>{xlabel}: %{{x:.3f}}"
                               f"<br>{ylabel}: %{{y:.3f}}<extra></extra>"),
            ))
        if not traces:
            continue

        x0, x1 = _padded(xlo, xhi, X_PAD_FRAC)
        if is_depth:
            # Anchor the depth axis to the window that was ASKED for, not to the
            # outermost reading. Bins sit at bin centres (10.907 ... 19.831), so a
            # 10–20 m window holds no reading at exactly 10 or 20; anchoring to the
            # data would push both those lines off the frame. Blank means surface
            # to deepest, which keeps 0 m visible for the same reason.
            top_req = depth_min if depth_min is not None else 0.0
            bot_req = depth_max if depth_max is not None else yhi
            pad = (bot_req - top_req) * Y_PAD_FRAC or 1.0
            lo, hi = top_req - pad, bot_req + pad
        else:
            lo, hi = _padded(ylo, yhi, Y_PAD_FRAC)
        yrange = [hi, lo] if invert else [lo, hi]   # descending → down is deeper

        head = f"{y_channel['name']} vs {s['name']}"
        if LOCATION:
            head = f"{LOCATION}: {head}"

        # Inverted, the profile is read downward from the surface, so the x axis
        # belongs at the top where the reader starts. Upright, it is an ordinary
        # graph and the x axis goes back to the bottom. Margins follow the axis,
        # then give or take room for the title, the legend and the y label.
        x_side = "top" if invert else "bottom"
        top_m, bottom_m = (110, 40) if invert else (70, 70)
        if not SHOW_TITLES:
            top_m -= 40
        if LEGEND_POS == "bottom":
            bottom_m += 40
        # Text widths are not known before drawing, so make room by character
        # count: 8 px each is on the safe side at this font size.
        est = lambda text: 8 * len(re.sub(r"<[^>]+>", "", str(text)))
        left_m = max(110, est(ylabel) + 16) if Y_LABEL_TOP else 70
        if LEGEND_POS == "left":
            # The legend's right edge lands at l - 0.2 * plot width, i.e. at
            # 1.2 * l - 146 px; pick l so the longest station name fits.
            need = 46 + est(max((t.name for t in traces), key=len))
            left_m = max(230, left_m, int((need + 146) / 1.2))

        if LEGEND_POS == "bottom":
            # Runs along under the plot, and sits lower when the x axis is down
            # there too, so the two do not overlap.
            legend = dict(orientation="h", x=0, xanchor="left", yanchor="top",
                          y=-0.03 if invert else -0.12)
        elif LEGEND_POS == "left":
            legend = dict(orientation="v", x=-0.2, xanchor="right", y=1, yanchor="top")
        else:
            legend = dict(orientation="v", x=1.02, xanchor="left", y=1, yanchor="top")
        legend["title"] = "Station"

        # The y label can sit upright above the axis, where the eye starts,
        # instead of rotated along it: an annotation, lifted clear of a
        # top-side x axis and its title.
        annotations = []
        if Y_LABEL_TOP:
            top_m += 24
            annotations.append(dict(
                text=ylabel, xref="paper", yref="paper", x=0, y=1,
                xanchor="right", yanchor="bottom", xshift=-6,
                yshift=40 if invert else 6, showarrow=False,
                font=dict(size=AXIS_FONT)))

        fig = go.Figure(traces)
        fig.update_layout(
            title=dict(text=head if SHOW_TITLES else "", x=0.5, xanchor="center",
                       font=dict(size=16)),
            xaxis=dict(range=[x0, x1], side=x_side,
                       title=dict(text=xlabel, font=dict(size=AXIS_FONT), standoff=8)),
            yaxis=dict(range=yrange,
                       title=dict(text="" if Y_LABEL_TOP else ylabel,
                                  font=dict(size=AXIS_FONT), standoff=8)),
            hovermode="closest", width=760, height=620,
            legend=legend, annotations=annotations,
            margin=dict(l=left_m, r=30, t=top_m, b=bottom_m),
            **_graph_theme(),
        )
        figs.append((s["name"], fig))
    return figs


# ─────────────── colour scales ───────────────
# cmocean colormaps, sampled to 16 stops each. Perceptually uniform and chosen
# per variable, so temperature reads as temperature across every figure.
#   Thyng, K.M., Greene, C.A., Hetland, R.D., Zimmerle, H.M. and DiMarco, S.F.
#   (2016). True colors of oceanography: guidelines for effective and accurate
#   colormap selection. Oceanography 29(3), 9–13.
#   https://doi.org/10.5670/oceanog.2016.66
#   Colormaps from https://github.com/kthyng/cmocean
#   MIT licence, Copyright (c) 2015 Kristen M. Thyng.
CMOCEAN = {
    "thermal": "#042333 #092e56 #1c3482 #40349f #5c3e9a #744992 #8b538d #a35b86 "
               "#bd637c #d66c6c #eb7958 #f78c45 #fca63c #fac140 #f3dd4b #e8fa5b",
    "haline":  "#2a186c #2e1e95 #1d37a1 #0d4e96 #125f8f #206e8b #2d7c89 #378b88 "
               "#409a86 #4aaa81 #5ab978 #71c86b #94d35d #bddc62 #e0e57a #fdef9a",
    "dense":   "#e6f1f1 #c9e3e8 #aed4e3 #96c5e2 #82b5e3 #76a4e5 #7390e3 #767cdc "
               "#7968ce #7954bb #7642a5 #71328c #682471 #5b1954 #4a133a #360e24",
    "algae":   "#d7f9d0 #c2eab7 #acdba0 #96cd8a #7ec175 #64b463 #44a855 #209c51 "
               "#098d4f #097d4b #126e45 #175f3d #1a5034 #19412b #173320 #122414",
    "turbid":  "#e9f6ab #dfe292 #d7cf7b #cfbc66 #c8a954 #bf9747 #b58740 #a8773c "
               "#9a6a3b #8a5e3a #795338 #674835 #563e30 #44342a #332a23 #221f1b",
    "deep":    "#fdfecc #dbf1b9 #b7e5ab #92d8a4 #71caa3 #5dbaa4 #52a8a3 #4b97a0 "
               "#45869c #407598 #3e6495 #3f528f #41407b #3c335f #332744 #281a2c",
    "matter":  "#feedb0 #fbd59a #f9be85 #f5a773 #f18f63 #eb7858 #e26253 #d64d54 "
               "#c63c59 #b32e5f #9f2462 #891d63 #721a60 #5b1758 #45144c #2f0f3e",
    "speed":   "#fffdcd #f3e9a9 #e7d684 #d8c55f #c4b73d #aaac20 #8ea20b #709707 "
               "#518c12 #32801f #187328 #0b632c #10542c #174327 #19331f #172313",
    "amp":     "#f1edec #e9d9d4 #e2c5bc #dcb1a3 #d69e8b #d08b73 #ca775b #c36346 "
               "#bc4e32 #b33826 #a62225 #941328 #7f0e29 #680f25 #520d1c #3c0912",
    "balance": "#181c43 #27337a #214cb6 #1670bc #438fba #75aabe #aac2cb #dbdee0 "
               "#e9d9d5 #dcb2a4 #d08b73 #c36346 #b33826 #941328 #680f25 #3c0912",
    "delta":   "#112040 #25367a #1b569d #2378a3 #3a98ab #6db6b3 #accec6 #e3ebde "
               "#f4e9aa #d9c560 #abac21 #709807 #33801f #0b642c #174327 #172313",
}

# oxy is the exception. cmocean's oxy has two hard edges, red below 0.2 of the
# range (hypoxic) and yellow above 0.8 (supersaturated). Sampling it evenly
# would blur them into a gradient, so its stops are listed with each edge
# doubled up, which is how Plotly draws a step.
OXY_STOPS = [[0, "#400505"], [0.098, "#6a060f"], [0.2, "#8f1808"], [0.2, "#504f4f"],
             [0.298, "#676666"], [0.4, "#81807f"], [0.498, "#9a9a99"],
             [0.6, "#b7b7b6"], [0.698, "#d4d4d3"], [0.8, "#f4f4f3"],
             [0.8, "#f8fe69"], [0.898, "#e7d82d"], [1, "#ddaf19"]]

# Which cmocean scheme suits which variable. A "_r" suffix reverses it — beam
# transmission is the inverse of turbidity, so it uses the turbid ramp backwards.
VARIABLE_COLORMAP = {
    "Temperature": "thermal", "Potential Temperature": "thermal",
    "Salinity": "haline", "Conductivity": "haline",
    "Density (sigma-t)": "dense",
    "Dissolved Oxygen": "oxy", "Oxygen Solubility": "oxy",
    "Oxygen Saturation": "oxy",
    "Fluorescence": "algae", "CDOM": "matter",
    "Turbidity": "turbid", "Beam Attenuation": "turbid",
    "Beam Transmission": "turbid_r",
    "Sound Velocity": "speed", "PAR": "amp",
    "Depth": "deep", "Pressure": "deep",
}


def colorscale(name):
    """cmocean scheme name -> Plotly colorscale. Append _r to reverse."""
    reverse = name.endswith("_r")
    key = name[:-2] if reverse else name
    if key == "oxy":
        # reversed, the edges still sit at 0.2 and 0.8 of the range
        if reverse:
            return [[round(1 - p, 3), c] for p, c in reversed(OXY_STOPS)]
        return [list(s) for s in OXY_STOPS]
    cols = CMOCEAN[key].split()
    if reverse:
        cols = cols[::-1]
    last = len(cols) - 1
    return [[i / last, c] for i, c in enumerate(cols)]


def variable_colorscale(variable_name):
    """Colour scale for a named variable, falling back to a neutral ramp."""
    return colorscale(VARIABLE_COLORMAP.get(variable_name, "deep"))


# Fixed colour ranges for the sections, so a colour always means the same value
# from one survey to the next rather than stretching to whatever was measured.
# (low, high, tick): the ends are round numbers and the tick lands the colour
# bar labels on round numbers too. The ranges follow long-term Puget Sound
# monitoring percentiles (King County offshore CTD, Ecology, PSEMP); values
# outside a range take the end colour. Oxygen depends on the unit the cast
# carries, and its ranges put the oxy map's red edge at the 2 mg/L hypoxia
# threshold and its yellow edge at 100 % saturation.
VARIABLE_RANGES = {
    "Temperature": (6.0, 20.0, 2.0),
    "Salinity": (20.0, 32.0, 2.0),
    "Density (sigma-t)": (18.0, 26.0, 1.0),
    "Dissolved Oxygen": {"mg/l": (0.0, 10.0, 2.0), "ml/l": (0.0, 7.0, 1.0),
                         "% sat": (0.0, 125.0, 25.0), "mol/kg": (0.0, 350.0, 50.0)},
    "Fluorescence": (0.0, 20.0, 2.0),
    "Beam Transmission": (50.0, 100.0, 10.0),
    "Turbidity": (0.0, 5.0, 1.0),
}


def variable_range(variable_name, unit=""):
    """(low, high, tick) for a named variable, or None when it has no fixed range."""
    r = VARIABLE_RANGES.get(variable_name)
    if isinstance(r, dict):
        u = (unit or "").lower()
        r = next((v for k, v in r.items() if k in u), None)
    return r


def nice_step(span, n=8):
    """A round step (1, 2, 2.5 or 5 times a power of ten) giving about n
    intervals across span. Used for colour bar ticks when a range is not one
    of the fixed ones above."""
    raw = max(abs(span), 1e-9) / n
    mag = 10 ** np.floor(np.log10(raw))
    return float(next((m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= raw), raw))


def _jsround(x):
    """Round half up (2.5 -> 3), as palette writers and JavaScript do; Python's
    own round() goes to the even neighbour."""
    return int(np.floor(float(x) + 0.5))


def _rgb_hex(r, g, b):
    """'#rrggbb' from 0-255 components, rounded and clamped."""
    return "#{:02x}{:02x}{:02x}".format(
        *(min(max(_jsround(v), 0), 255) for v in (r, g, b)))


def _hex_rgb(c):
    """'#rrggbb' -> (r, g, b)."""
    return tuple(int(c[i:i + 2], 16) for i in (1, 3, 5))


def _surfer_color(token):
    """Surfer writes colours as a name or as 'Rxxx Gyyy Bzzz [Aaaa]'. The alpha
    is dropped: Plotly colorscales carry no opacity."""
    t = str(token).strip().strip('"').strip()
    m = re.match(r"R\s*(\d+)\s+G\s*(\d+)\s+B\s*(\d+)", t, re.I)
    if m:
        return _rgb_hex(*(int(g) for g in m.groups()))
    return _named_color(t)


# Golden Software Surfer's named colours, all 104, keyed lower-case with the
# spaces removed ("Deep Navy Blue" -> "deepnavyblue").
_SURFER_NAMED = {
    "black": "#000000", "90%black": "#191919", "80%black": "#333333",
    "70%black": "#4d4d4d", "60%black": "#666666", "50%black": "#808080",
    "40%black": "#999999", "30%black": "#b3b3b3", "20%black": "#cccccc",
    "10%black": "#e6e6e6", "white": "#ffffff", "blue": "#0000ff",
    "cyan": "#00ffff", "green": "#00ff00", "yellow": "#ffff00",
    "red": "#ff0000", "magenta": "#ff00ff", "purple": "#9900cc",
    "orange": "#ff6600", "pink": "#ff99cc", "darkbrown": "#663333",
    "powderblue": "#ccccff", "pastelblue": "#9999ff", "babyblue": "#6699ff",
    "electricblue": "#6666ff", "twilightblue": "#6666cc", "navyblue": "#003399",
    "deepnavyblue": "#000066", "desertblue": "#336699", "dodgerblue": "#1389ff",
    "skyblue": "#00ccff", "iceblue": "#99ffff", "smaltblue": "#0068d0",
    "lightbluegreen": "#99cccc", "oceangreen": "#669999", "mossgreen": "#336666",
    "darkgreen": "#003333", "forestgreen": "#006633", "grassgreen": "#009933",
    "wildwillow": "#b5cc61", "kentuckygreen": "#339966", "lightgreen": "#33cc66",
    "springgreen": "#33cc33", "turquoise": "#66ffcc", "seagreen": "#33cc99",
    "fadedgreen": "#99cc99", "ghostgreen": "#ccffcc", "mintgreen": "#99ff99",
    "armygreen": "#669966", "avocadogreen": "#669933", "martiangreen": "#99cc33",
    "dullgreen": "#99cc66", "chartreuse": "#99ff00", "moongreen": "#ccff66",
    "murkygreen": "#333300", "olivedrab": "#666633", "khaki": "#999966",
    "olive": "#999933", "bananayellow": "#cccc33", "lightyellow": "#ffff66",
    "chalk": "#ffff99", "paleyellow": "#ffffcc", "brown": "#996633",
    "redbrown": "#cc6633", "gold": "#cc9933", "autumnorange": "#ff6633",
    "lightorange": "#ff9933", "peach": "#ff9966", "deepyellow": "#ffcc00",
    "sand": "#ffcc99", "walnut": "#663300", "rubyred": "#990000",
    "brickred": "#cc3300", "tropicalpink": "#ff6666", "softpink": "#ff9999",
    "fadedpink": "#ffcccc", "darkred": "#800000", "crimson": "#993366",
    "regalred": "#cc3366", "deeprose": "#cc3399", "neonred": "#ff0066",
    "deeppink": "#ff6699", "hotpink": "#ff3399", "dustyrose": "#cc6699",
    "plum": "#660066", "deepviolet": "#990099", "lightviolet": "#ff99ff",
    "violet": "#cc66cc", "dustyplum": "#996699", "palepurple": "#cc99cc",
    "majesticpurple": "#9933cc", "neonpurple": "#cc33ff", "lightpurple": "#cc66ff",
    "twilightviolet": "#9966cc", "easterpurple": "#cc99ff", "deeppurple": "#330066",
    "grape": "#663399", "blueviolet": "#9966ff", "bluepurple": "#9900ff",
    "deepriver": "#6600cc", "deepazure": "#6633ff", "stormblue": "#330099",
    "deepblue": "#3300cc", "darkblue": "#000080",
}

# The usual web and GRASS names as well, for GMT tables and value lists. Where
# the two disagree (olive, gold, brown, violet) the web value wins.
_WEB_NAMED = {
    "grey": "#808080", "gray": "#808080", "lightgray": "#c0c0c0", "lightgrey": "#c0c0c0",
    "darkgray": "#404040", "darkgrey": "#404040", "navy": "#000080", "olive": "#808000",
    "teal": "#008080", "maroon": "#800000", "aqua": "#00ffff", "lime": "#00ff00",
    "violet": "#ee82ee", "indigo": "#4b0082", "gold": "#ffd700", "silver": "#c0c0c0",
    "brown": "#a52a2a",
}
_NAMED_COLORS = {**_SURFER_NAMED, **_WEB_NAMED}


def _named_color(token):
    """Hex colour for a colour name, or None. Case and spaces do not matter;
    GMT's gray0..gray100 are made by rule."""
    k = re.sub(r"\s+", "", str(token)).lower()
    m = re.fullmatch(r"gr[ae]y([0-9]{1,3})", k)
    if m:
        v = round(min(int(m.group(1)), 100) * 2.55)
        return _rgb_hex(v, v, v)
    return _NAMED_COLORS.get(k)


def parse_clr(text):
    """Read a Surfer .clr colour spectrum into a Plotly colorscale.

    Header 'ColorMap [Version] [InterpMethod] [ColorNodes] [OpacityNodes]', then
    anchor lines 'Position(0-100) R G B [Alpha]'. Version 3 adds separate
    opacity anchors after the colour ones; Plotly colorscales carry no alpha, so
    those are read and ignored.
    https://surferhelp.goldensoftware.com/topics/color_spectrum_file_format.htm

    Returns (colorscale, note) or (None, reason)."""
    lines = [l.strip() for l in str(text).splitlines() if l.strip()]
    if not lines:
        return None, "file is empty"
    head = lines[0].split()
    if not head or head[0].upper() != "COLORMAP":
        return None, "first line is not a ColorMap header"
    version = int(head[1]) if len(head) > 1 and head[1].isdigit() else 1
    colour_nodes = int(head[3]) if len(head) > 3 and head[3].isdigit() else None

    stops = []
    for line in lines[1:]:
        nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", line)
        if len(nums) < 4:
            continue                      # opacity anchors have only two
        pos = float(nums[0])
        r, g, b = (int(float(n)) for n in nums[1:4])
        if not all(0 <= v <= 255 for v in (r, g, b)):
            return None, f"colour out of range on: {line[:40]}"
        stops.append((pos, "#{:02x}{:02x}{:02x}".format(r, g, b)))
        if colour_nodes and len(stops) >= colour_nodes:
            break

    if len(stops) < 2:
        return None, "need at least two colour anchors"
    stops.sort(key=lambda s: s[0])
    lo, hi = stops[0][0], stops[-1][0]
    if hi <= lo:
        return None, "anchor positions do not increase"
    scale = [[(p - lo) / (hi - lo), c] for p, c in stops]
    return scale, f"v{version}, {len(stops)} colour anchors"


def parse_lvl(text):
    """Read a Surfer .lvl level file: contour levels and their fill colours.

    Three forms. Bare numbers are levels only. 'LVL2' gives
    Level, Flags, LColor, LStyle, LWidth, FFGColor, FBGColor, FPattern, FMode,
    and 'LVL3' extends that with pattern placement. Fields are separated by
    commas or whitespace, colours are quoted names or 'Rxxx Gyyy Bzzz [Aaaa]',
    and a single quote starts a comment.
    https://surferhelp.goldensoftware.com/topics/level_file_format.htm

    Returns (levels, colorscale_or_None, note) or (None, None, reason)."""
    raw = [l.rstrip() for l in str(text).splitlines()]
    if not raw:
        return None, None, "file is empty"

    fmt = "LVL1"
    body = []
    for line in raw:
        s = line.strip()
        if not s:
            continue
        if s.upper() in ("LVL2", "LVL3"):
            fmt = s.upper()
            continue
        body.append(s)

    levels, colours = [], []
    for line in body:
        # A single quote opens a comment, but also quotes colour and style names,
        # so only treat it as a comment when it is not part of a quoted field.
        quoted = re.findall(r'"([^"]*)"', line)
        stripped = re.sub(r'"[^"]*"', '""', line)
        if stripped.lstrip().startswith("'"):
            continue
        stripped = stripped.split("'")[0]
        nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", stripped)
        if not nums:
            continue
        levels.append(float(nums[0]))
        # LVL2/3 field order puts the fill foreground colour after line colour,
        # line style and width: quoted fields are LColor, LStyle, FFGColor, ...
        fill = None
        if fmt in ("LVL2", "LVL3") and len(quoted) >= 3:
            fill = _surfer_color(quoted[2])
        elif quoted:
            fill = _surfer_color(quoted[0])
        colours.append(fill)

    if not levels:
        return None, None, "no levels found"
    if len(levels) > 1 and any(b <= a for a, b in zip(levels, levels[1:])):
        return None, None, "levels must increase"

    scale = None
    if len(levels) > 1 and all(c for c in colours):
        lo, hi = levels[0], levels[-1]
        scale = [[(v - lo) / (hi - lo), c] for v, c in zip(levels, colours)]
    note = f"{fmt}, {len(levels)} levels"
    note += ", fill colours read" if scale else ", levels only"
    return levels, scale, note


# ─────────────── palette files ───────────────
# A palette file of your own can colour the sections in cell 4. Read here:
#   .clr  Surfer colour spectrum (positions 0-100, R G B [A]); an ESRI .clr
#         of "index r g b" lines is read as a value list
#   .lvl  Surfer level file: contour levels with fill colours (LVL2/LVL3) or
#         levels only (LVL1); the levels are real values and set the range
#   .cpt  GMT colour palette table (z0 colour z1 colour ..., B/F/N lines)
#   .pal  Ocean Data View palette (indexed colour components) or an ncWMS
#         palette (one hex colour per line)
#   .spk  Ferret spectrum: "setpoint r g b" with colours 0-100 and an
#         RGB_Mapping header saying whether set points are percentages,
#         real values or level indices
#   .rgb  NCL colour table: "r g b" per line, evenly spaced
#   .cpd  SNAP / SeaDAS colour palette definition (colorN=, sampleN=)
#   .ggr  GIMP gradient
#   .json ParaView / VTK colour map presets (RGBPoints)
#   .xml  QGIS style colour ramp, or the older ParaView ColorMaps.xml
#   .txt/.csv/.rules  "value r g b [a]" lines as QGIS, GRASS and GDAL write
#         them; values are real like .lvl, or percentages
# A palette whose values are real (levels) also sets the colour range, so it
# belongs to one variable; a position-only palette can colour every section.
# Opacity in any of them is dropped, since Plotly colorscales carry none.
PALETTE_EXTENSIONS = [".clr", ".lvl", ".cpt", ".pal", ".spk", ".rgb", ".cpd",
                      ".ggr", ".json", ".xml", ".txt", ".csv", ".rules"]


def _lead_float(s):
    """The number a token starts with ('50%' -> 50.0, '-8000' -> -8000.0), or
    None when it starts with something else."""
    m = re.match(r"\s*[-+]?([0-9]+\.?[0-9]*|\.[0-9]+)([eE][-+]?[0-9]+)?", str(s))
    return float(m.group(0)) if m else None


def _whole_float(s):
    """A token that is nothing but a number, or None."""
    t = str(s).strip()
    if re.fullmatch(r"[-+]?([0-9]+\.?[0-9]*|\.[0-9]+)([eE][-+]?[0-9]+)?", t):
        return float(t)
    return None


def _num_text(v):
    """A level for a note: 8 rather than 8.0, 0.2 rather than 0.2000000001."""
    v = float(v)
    return str(int(v)) if v.is_integer() and abs(v) < 1e15 else f"{v:.6g}"


def _strip_comment(line):
    """A comment runs from a '#' or "'" at the start of the line or after a
    space; a '#' glued to six hex digits is a colour, not a comment."""
    return re.sub(r"(^|\s)[#'](?![0-9a-fA-F]{6}\b).*$", "", line).strip()


def _hsv_hex(h, s, v):
    """Hue in degrees with s and v 0-1 -> '#rrggbb'."""
    c = v * s
    x = c * (1 - abs(np.fmod(h / 60, 2) - 1))
    m = v - c
    r, g, b = ((c, x, 0) if h < 60 else (x, c, 0) if h < 120 else (0, c, x) if h < 180
               else (0, x, c) if h < 240 else (x, 0, c) if h < 300 else (c, 0, x))
    return _rgb_hex((r + m) * 255, (g + m) * 255, (b + m) * 255)


def _hex_hsv(c):
    r, g, b = (v / 255 for v in _hex_rgb(c))
    mx, mn = max(r, g, b), min(r, g, b)
    d = mx - mn
    h = 0.0
    if d:
        h = (np.fmod((g - b) / d, 6) if mx == r
             else (b - r) / d + 2 if mx == g else (r - g) / d + 4)
    h = (h * 60 + 360) % 360
    return h, (d / mx if mx else 0.0), mx


def _blend_hsv(a, b, t):
    """The colour t of the way from a to b, blending in HSV."""
    h0, s0, v0 = _hex_hsv(a)
    h1, s1, v1 = _hex_hsv(b)
    return _hsv_hex(h0 + (h1 - h0) * t, s0 + (s1 - s0) * t, v0 + (v1 - v0) * t)


def _dedupe_stops(stops):
    """Drop repeated stops and make sure the scale runs from 0 to 1."""
    out = []
    for p, c in stops:
        if not out or out[-1][0] != p or out[-1][1] != c:
            out.append([p, c])
    if out[0][0] != 0:
        out.insert(0, [0.0, out[0][1]])
    if out[-1][0] != 1:
        out.append([1.0, out[-1][1]])
    return out


def _from_positions(stops, note):
    """A palette given as positions 0-1: no levels, so no colour range."""
    if len(stops) < 2:
        raise ValueError(f"{note}: fewer than two colours")
    stops = sorted(stops, key=lambda s: s[0])
    return {"stops": _dedupe_stops(stops), "levels": None, "note": note}


def _from_levels(levels, colours, note, kind):
    """A palette given as real values: the first and last set the colour range.
    No colours (a bare .lvl) gives stops None, so the variable keeps its own."""
    if len(levels) < 2:
        raise ValueError(f"{kind}: fewer than two levels")
    lo, hi = levels[0], levels[-1]
    if not hi > lo:
        raise ValueError(f"{kind}: levels must increase")
    stops = ([[(v - lo) / (hi - lo), c] for v, c in zip(levels, colours)]
             if colours else None)
    return {"stops": stops, "levels": [float(v) for v in levels], "note": note}


def _range_note(kind, n, lo, hi):
    return f"{kind}, {n} levels; colour range set to {_num_text(lo)} to {_num_text(hi)}"


def _read_clr(text):
    scale, note = parse_clr(text)
    if scale is None:
        raise ValueError(f".clr: {note}")
    return {"stops": scale, "levels": None, "note": f".clr {note}"}


def _read_lvl(text):
    levels, scale, note = parse_lvl(text)
    if levels is None:
        raise ValueError(f".lvl: {note}")
    if len(levels) < 2:
        raise ValueError(".lvl: fewer than two levels found")
    fmt, lo, hi = note.split(",")[0], _num_text(levels[0]), _num_text(levels[-1])
    if scale:
        note = _range_note(f"{fmt} with fill colours", len(levels), levels[0], levels[-1])
    else:
        note = (f"{fmt}, {len(levels)} levels without colours; colour range set to "
                f"{lo} to {hi}, colours stay the default")
    return {"stops": scale, "levels": [float(v) for v in levels], "note": note}


def _read_cpt(text):
    """GMT .cpt: "z0 colour z1 colour [A|U|L]" lines, the colour as r/g/b,
    r g b, #rrggbb, a gray value, h-s-v (or r/g/b with COLOR_MODEL = HSV) or a
    name. B, F and N lines (background, foreground, NaN) are ignored."""
    hsv = re.search(r"COLOR_MODEL\s*=\s*\+?HSV", text, re.I) is not None

    def colour(tok, i):
        if i >= len(tok):
            raise ValueError(".cpt: line too short")
        t = tok[i]
        if re.fullmatch(r"#[0-9a-fA-F]{6}", t):
            return t.lower(), 1
        if re.fullmatch(r"[0-9.]+/[0-9.]+/[0-9.]+", t):
            a, b, c = (float(x) for x in t.split("/"))
            return (_hsv_hex(a, b, c) if hsv else _rgb_hex(a, b, c)), 1
        if re.fullmatch(r"[0-9.]+-[0-9.]+-[0-9.]+", t):
            h, s, v = (float(x) for x in t.split("-"))
            return _hsv_hex(h, s, v), 1
        if (re.fullmatch(r"[0-9]+", t) and i + 2 < len(tok)
                and re.fullmatch(r"[0-9]+", tok[i + 1]) and re.fullmatch(r"[0-9]+", tok[i + 2])):
            return _rgb_hex(int(t), int(tok[i + 1]), int(tok[i + 2])), 3
        if re.fullmatch(r"[0-9]+(\.[0-9]+)?", t):
            return _rgb_hex(float(t), float(t), float(t)), 1
        n = _named_color(t)
        if n:
            return n, 1
        raise ValueError(f'.cpt: cannot read colour "{t}"')

    segs = []
    for raw in text.splitlines():
        s = _strip_comment(raw)
        if not s or re.match(r"[BFN]\b", s, re.I):
            continue
        tok = s.split()
        if len(tok) < 4 or _lead_float(tok[0]) is None:
            continue
        z0 = _lead_float(tok[0])
        c0, n0 = colour(tok, 1)
        z1 = _lead_float(tok[1 + n0]) if 1 + n0 < len(tok) else None
        if z1 is None:
            continue
        c1, _n = colour(tok, 2 + n0)
        segs.append((z0, c0, z1, c1))
    if not segs:
        raise ValueError(".cpt: no colour segments found")
    segs.sort(key=lambda g: g[0])
    lo, hi = segs[0][0], segs[-1][2]
    if not hi > lo:
        raise ValueError(".cpt: z values must increase")
    stops = []
    for z0, c0, z1, c1 in segs:
        p0, p1 = (z0 - lo) / (hi - lo), (z1 - lo) / (hi - lo)
        stops.append([p0, c0])
        # an HSV table blends round the hue circle, which a straight RGB blend
        # between the two end colours would miss; sample the segment instead
        if hsv and c0 != c1:
            for k in range(1, 12):
                stops.append([p0 + (p1 - p0) * k / 12, _blend_hsv(c0, c1, k / 12)])
        stops.append([p1, c1])
    return {"stops": _dedupe_stops(stops), "levels": [lo, hi],
            "note": _range_note(".cpt", len(segs) + 1, lo, hi)}


def _read_value_rgb(text):
    """"value r g b [a]" or "value,r,g,b[,a][,label]" per line, as QGIS, GRASS
    and GDAL write them. nv, default and end lines are skipped; r:g:b, hex and
    colour names are accepted; "10%" values make a position palette instead of
    a range."""
    vals, colours = [], []
    percent = plain = 0
    for raw in text.splitlines():
        s = _strip_comment(raw)
        if not s or re.match(r"(nv|default|end|INTERPOLATION)\b", s, re.I):
            continue
        p = re.split(r"[\s,;:]+", s)
        if len(p) < 2:
            continue
        v = _lead_float(p[0])
        if v is None:
            continue
        if len(p) >= 4 and all(_whole_float(x) is not None for x in p[1:4]):
            col = _rgb_hex(*(float(x) for x in p[1:4]))
        elif re.fullmatch(r"#[0-9a-fA-F]{6}", p[1]):
            col = p[1].lower()
        else:
            col = _named_color(p[1])
        if not col:
            continue
        vals.append(v)
        colours.append(col)
        if p[0].endswith("%"):
            percent += 1
        else:
            plain += 1
    if len(vals) < 2:
        raise ValueError('not a palette: expected lines of "value r g b"')
    order = sorted(range(len(vals)), key=lambda i: vals[i])
    v = [vals[i] for i in order]
    c = [colours[i] for i in order]
    if percent and not plain:
        return _from_positions([[min(max(x / 100, 0), 1), col] for x, col in zip(v, c)],
                               f"{len(v)} colours at percentages")
    return _from_levels(v, c, _range_note("value list", len(v), v[0], v[-1]), "palette")


def _read_spk(text):
    """Ferret .spk: "setpoint r g b" with colours 0-100. RGB_Mapping Percent
    (the default) makes the set points positions, By_value real values, and
    By_level level indices, taken as evenly spaced."""
    m = re.search(r"RGB_Mapping\s+(Percent|By_value|By_level)", text, re.I)
    mode = (m.group(1) if m else "Percent").lower()
    pts = []
    for raw in text.splitlines():
        s = raw.split("!")[0].strip()
        if not s or re.search(r"RGB_Mapping", s, re.I):
            continue
        p = [_whole_float(x) for x in s.split()]
        if len(p) < 4 or any(x is None for x in p):
            continue
        pts.append((p[0], _rgb_hex(p[1] * 2.55, p[2] * 2.55, p[3] * 2.55)))
    if len(pts) < 2:
        raise ValueError(".spk: no colour lines found")
    pts.sort(key=lambda q: q[0])
    n = len(pts)
    if mode == "by_value":
        return _from_levels([q[0] for q in pts], [q[1] for q in pts],
                            _range_note(".spk By_value", n, pts[0][0], pts[-1][0]), ".spk")
    if mode == "by_level":
        return _from_positions([[i / (n - 1), q[1]] for i, q in enumerate(pts)],
                               f".spk By_level, {n} colours evenly spaced")
    return _from_positions([[min(max(q[0] / 100, 0), 1), q[1]] for q in pts],
                           f".spk, {n} colours")


def _rgb_rows(rows, note):
    """Evenly spaced colours; components given as 0-1 are scaled up to 0-255."""
    if len(rows) < 2:
        raise ValueError(f"{note}: fewer than two colours")
    k = 255 if max(v for r in rows for v in r) <= 1 else 1
    n = len(rows)
    stops = [[i / (n - 1), _rgb_hex(r[0] * k, r[1] * k, r[2] * k)] for i, r in enumerate(rows)]
    return {"stops": stops, "levels": None, "note": note}


def _read_odv_pal(text):
    """Ocean Data View .pal: 177 rows of "index r g b" as fractions; indices
    0-31 are the program's own interface colours and 32-144 are the ramp."""
    rows = []
    for raw in text.splitlines():
        p = [_whole_float(x) for x in re.split(r"[\s,]+", raw.strip())]
        if len(p) in (3, 4) and all(x is not None for x in p):
            rows.append(p)
    if len(rows) < 8:
        raise ValueError(".pal: too few colour rows")
    indexed = all(len(r) == 4 and r[0].is_integer() for r in rows)
    if indexed and len(rows) >= 145:
        ramp = [r[1:] for r in rows if 32 <= r[0] <= 144]
    else:
        ramp = [r[1:] if len(r) == 4 else r for r in rows]
    return _rgb_rows(ramp, f"ODV .pal, {len(ramp)} colours")


def _read_rgb_list(text):
    """NCL-style .rgb: "r g b" per line with an optional "ncolors=" header, as
    0-255 or 0-1; also plain lists of one hex colour per line (ncWMS and the
    like)."""
    rows = []
    for raw in text.splitlines():
        s = _strip_comment(re.sub(r";.*$", "", raw))
        if not s or "=" in s:
            continue
        m = re.fullmatch(r"#?([0-9a-fA-F]{6})", s)
        if m:
            rows.append(list(_hex_rgb("#" + m.group(1))))
            continue
        p = [_whole_float(x) for x in re.split(r"[\s,]+", s)]
        if len(p) in (3, 4) and all(x is not None for x in p):
            rows.append(p[:3])
    if len(rows) < 2:
        raise ValueError('.rgb: expected "r g b" or hex lines')
    return _rgb_rows(rows, f"{len(rows)} colours, evenly spaced")


def _read_cpd(text):
    """SNAP / SeaDAS .cpd: numPoints=N, colorI=r,g,b[,a], sampleI=value; with
    autoDistribute=true the samples are relative, otherwise real values."""
    def get(key):
        m = re.search(rf"^\s*{key}\s*=\s*(.+)$", text, re.M)
        return m.group(1).strip() if m else None

    n = _lead_float(get("numPoints") or "")
    n = int(n) if n is not None else 0
    if not n > 1:
        raise ValueError(".cpd: numPoints missing")
    colours, samples = [], []
    for i in range(n):
        c = [_whole_float(x) for x in (get(f"color{i}") or "").split(",")]
        s = _lead_float(get(f"sample{i}") or "")
        if len(c) < 3 or any(x is None for x in c) or s is None:
            raise ValueError(f".cpd: point {i} incomplete")
        colours.append(_rgb_hex(c[0], c[1], c[2]))
        samples.append(s)
    if re.search(r"autoDistribute\s*=\s*true", text, re.I):
        lo, hi = samples[0], samples[-1]
        return _from_positions([[(s - lo) / ((hi - lo) or 1), c] for s, c in zip(samples, colours)],
                               f".cpd, {n} colours (auto-distributed)")
    return _from_levels(samples, colours, _range_note(".cpd", n, samples[0], samples[-1]), ".cpd")


def _read_ggr(text):
    """GIMP .ggr: "GIMP Gradient", "Name:", a count, then one segment per line:
    left mid right, left RGBA, right RGBA, blend, colouring [, endpoint types]."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines or not lines[0].startswith("GIMP Gradient"):
        raise ValueError(".ggr: missing GIMP Gradient header")
    stops = []
    for l in lines[2:]:
        p = [_whole_float(x) for x in l.split()]
        if len(p) < 11 or any(x is None for x in p):
            continue
        stops.append([p[0], _rgb_hex(p[3] * 255, p[4] * 255, p[5] * 255)])
        stops.append([p[2], _rgb_hex(p[7] * 255, p[8] * 255, p[9] * 255)])
    return _from_positions(stops, f".ggr, {len(stops) // 2} segments")


def _read_paraview(text):
    """ParaView / VTK colour map JSON: [{"Name", "RGBPoints": [x, r, g, b, ...]}]
    with r g b as 0-1; the x values are the preset's own units, taken as real."""
    import json
    try:
        # ParaView's own ColorMaps.json carries trailing commas
        doc = json.loads(re.sub(r",(\s*[\]}])", r"\1", text))
    except ValueError:
        raise ValueError("not JSON")
    maps = doc if isinstance(doc, list) else [doc]
    m = next((x for x in maps if isinstance(x, dict) and isinstance(x.get("RGBPoints"), list)), None)
    if m is None:
        raise ValueError(".json: no RGBPoints colour map found")
    pts = m["RGBPoints"]
    if not all(isinstance(v, (int, float)) for v in pts):
        raise ValueError(".json: RGBPoints must be numbers")
    levels, colours = [], []
    for i in range(0, len(pts) - 3, 4):
        levels.append(float(pts[i]))
        colours.append(_rgb_hex(pts[i + 1] * 255, pts[i + 2] * 255, pts[i + 3] * 255))
    if len(levels) < 2:
        raise ValueError(".json: fewer than two levels")
    name = m.get("Name") if m.get("Name") is not None else "colour map"
    return _from_levels(levels, colours,
                        _range_note(f'ParaView "{name}"', len(levels), levels[0], levels[-1]), ".json")


def _read_paraview_xml(text):
    """The older ParaView ColorMaps.xml: <ColorMap name=...><Point x= r= g= b=/>."""
    m = re.search(r"<ColorMap\b[^>]*>.*?</ColorMap>", text, re.I | re.S)
    if not m:
        raise ValueError(".xml: no ColorMap found")
    block = m.group(0)
    nm = re.search(r'name="([^"]*)"', block)
    name = nm.group(1) if nm else "colour map"
    levels, colours = [], []
    for p in re.finditer(r"<Point\b([^>]*)/>", block):
        attrs = p.group(1)

        def attr(k):
            mm = re.search(rf'\b{k}="([^"]*)"', attrs)
            return _lead_float(mm.group(1)) if mm else None

        x, r, g, b = attr("x"), attr("r"), attr("g"), attr("b")
        if None in (x, r, g, b):
            continue
        levels.append(x)
        colours.append(_rgb_hex(r * 255, g * 255, b * 255))
    if len(levels) < 2:
        raise ValueError(".xml: fewer than two levels")
    return _from_levels(levels, colours,
                        _range_note(f'ParaView "{name}"', len(levels), levels[0], levels[-1]), ".xml")


def _read_qgis_xml(text):
    """QGIS style XML: <colorramp type="gradient"> with color1, color2 and
    "stops" as offset;r,g,b,a entries joined by ':'."""
    m = re.search(r'<colorramp[^>]*type="gradient"[^>]*>.*?</colorramp>', text, re.I | re.S)
    if not m:
        raise ValueError(".xml: no gradient colorramp found")
    ramp = m.group(0)

    def val(key):
        mm = re.search(rf'(?:k="{key}"\s+v="([^"]*)"|name="{key}"\s+(?:type="[^"]*"\s+)?value="([^"]*)")',
                       ramp)
        if not mm:
            return None
        return mm.group(1) if mm.group(1) is not None else mm.group(2)

    def rgba(s):
        p = [_whole_float(x) for x in s.split(",")]
        if len(p) < 3 or any(x is None for x in p[:3]):
            raise ValueError(f'.xml: cannot read colour "{s}"')
        return _rgb_hex(p[0], p[1], p[2])

    c1, c2 = val("color1"), val("color2")
    if not c1 or not c2:
        raise ValueError(".xml: colorramp lacks color1/color2")
    stops = [[0.0, rgba(c1)]]
    for e in (val("stops") or "").split(":"):
        parts = e.split(";")
        if len(parts) > 1 and parts[1] and _lead_float(parts[0]) is not None:
            stops.append([_lead_float(parts[0]), rgba(parts[1])])
    stops.append([1.0, rgba(c2)])
    return _from_positions(stops, f"QGIS ramp, {len(stops)} colours")


# Readers to try for each extension, in order, after the header sniff.
_PALETTE_READERS = {
    ".clr": [_read_clr, _read_value_rgb], ".lvl": [_read_lvl], ".cpt": [_read_cpt],
    ".pal": [_read_odv_pal, _read_rgb_list, _read_value_rgb], ".spk": [_read_spk],
    ".rgb": [_read_rgb_list, _read_value_rgb], ".cpd": [_read_cpd], ".ggr": [_read_ggr],
    ".json": [_read_paraview], ".xml": [_read_qgis_xml, _read_paraview_xml],
    ".txt": [_read_value_rgb, _read_cpt, _read_rgb_list], ".csv": [_read_value_rgb],
    ".rules": [_read_value_rgb],
}


def read_palette(text, filename):
    """Read a colour palette file (the formats listed above PALETTE_EXTENSIONS)
    into {"stops": Plotly colorscale or None, "levels": [values] or None,
    "note": str}.

    stops is None for a levels-only file (a bare Surfer .lvl): the variable
    keeps its own colours and just the range changes. levels is None for a
    palette given as positions, which fits any variable. The reader is picked
    by sniffing the header, then the extension, then whatever else fits.
    Raises ValueError when nothing reads it."""
    text = str(text).lstrip("\ufeff")
    ext = re.search(r"\.[^.]+$", str(filename))
    ext = ext.group(0).lower() if ext else ""
    head = text[:300]
    readers = []
    if re.match(r"\s*ColorMap", head, re.I):
        readers.append(_read_clr)
    if re.search(r"^\s*LVL[123]\b", head, re.I | re.M):
        readers.append(_read_lvl)
    if re.search(r"COLOR_MODEL", head, re.I):
        readers.append(_read_cpt)
    if re.match(r"\s*GIMP Gradient", head):
        readers.append(_read_ggr)
    if re.search(r"numPoints\s*=", head):
        readers.append(_read_cpd)
    if re.match(r"\s*RGB_Mapping", head, re.I):
        readers.append(_read_spk)
    if re.match(r"\s*[\[{]", head):
        readers.append(_read_paraview)
    if re.search(r"<colorramp", text, re.I):
        readers.append(_read_qgis_xml)
    if re.search(r"<ColorMap\b", text):
        readers.append(_read_paraview_xml)
    readers += _PALETTE_READERS.get(ext, []) + [_read_value_rgb, _read_cpt, _read_rgb_list]
    errors = []
    for reader in dict.fromkeys(readers):
        try:
            out = reader(text)
        except Exception as e:          # a reader given the wrong format fails any way it likes
            errors.append(str(e))
            continue
        stops = out["stops"]
        if stops and (stops[0][0] < 0 or stops[-1][0] > 1
                      or any(b[0] < a[0] for a, b in zip(stops, stops[1:]))):
            errors.append("colour positions do not increase")
            continue
        return out
    raise ValueError(f"{filename}: not a palette this notebook reads "
                     f"({', '.join(PALETTE_EXTENSIONS)}). {errors[0]}")


_HEMISPHERE = {"N": 1, "S": -1, "E": 1, "W": -1}


def parse_coordinate(text, kind="lat"):
    """Read a latitude or longitude written almost any way.

    Handles decimal degrees, degrees with decimal minutes, and degrees minutes
    seconds; degree, minute and second marks in ASCII or Unicode; a hemisphere
    letter anywhere in the string or a leading minus; comma decimal separators;
    and raw NMEA ddmm.mmm. Any number of decimal places.

        47.4012          47 24.072 N        47°24'04.32"N      -122 31 51.6
        47,4012          N 47 24.072        122:31:51.6 W      4724.072

    Returns (decimal_degrees, description) on success, or (None, reason)."""
    if text is None:
        return None, "empty"
    s = str(text).strip().upper()
    if not s:
        return None, "empty"

    for mark in ("°", "′", "″", "’", "‘", "´",
                 "ʼ", "'", '"', ":", "_"):
        s = s.replace(mark, " ")

    letters = {c for c in s if c in "NSEW"}
    if len(letters) > 1:
        return None, "more than one hemisphere letter"
    hemi = letters.pop() if letters else None
    if hemi and kind == "lat" and hemi in "EW":
        return None, f"{hemi} is a longitude direction"
    if hemi and kind == "lon" and hemi in "NS":
        return None, f"{hemi} is a latitude direction"

    negative = bool(re.match(r"\s*-", s))
    s = re.sub(r"[NSEW+\-]", " ", s)
    s = re.sub(r"(?<=\d),(?=\d)", ".", s)      # 47,4012 -> 47.4012
    s = s.replace(",", " ")

    if re.search(r"[A-Z]", s):
        return None, "unexpected letters"
    parts = re.findall(r"\d+(?:\.\d+)?", s)
    if not parts:
        return None, "no numbers found"
    if len(parts) > 3:
        return None, "too many numbers"

    vals = [float(p) for p in parts]
    # Only the last component may be fractional: "47 24.072" is fine, "47.4.012"
    # is a typo rather than degrees-and-minutes.
    if len(vals) > 1 and any(v != int(v) for v in vals[:-1]):
        return None, "only the last number may have decimals"
    minutes = vals[1] if len(vals) > 1 else 0.0
    seconds = vals[2] if len(vals) > 2 else 0.0
    if minutes >= 60:
        return None, "minutes must be under 60"
    if seconds >= 60:
        return None, "seconds must be under 60"

    magnitude = vals[0] + minutes / 60.0 + seconds / 3600.0
    form = ("decimal degrees", "degrees + decimal minutes",
            "degrees minutes seconds")[len(vals) - 1]
    limit = 90.0 if kind == "lat" else 180.0

    # A bare number too large to be degrees may be raw NMEA, which packs degrees
    # and minutes together: 4724.072 means 47° 24.072'.
    if len(vals) == 1 and magnitude > limit:
        deg, mins = divmod(vals[0], 100.0)
        if mins < 60 and deg + mins / 60.0 <= limit:
            magnitude, form = deg + mins / 60.0, "NMEA ddmm.mmm"

    sign = _HEMISPHERE[hemi] if hemi else (-1 if negative else 1)
    value = sign * magnitude
    if abs(value) > limit:
        return None, f"outside ±{limit:g}°"
    return value, form


def format_coordinate(value, kind="lat"):
    """Decimal degrees plus the same position in degrees minutes seconds."""
    if value is None:
        return "—"
    hemi = ("N" if value >= 0 else "S") if kind == "lat" else ("E" if value >= 0 else "W")
    a = abs(value)
    d = int(a)
    m = int((a - d) * 60)
    sec = (a - d - m / 60.0) * 3600.0
    return f"{value:.6f}°  ({d}° {m}' {sec:.2f}\" {hemi})"


# Positions for the casts shipped in example_data/, from the field sheet for
# 15 May 2026 (GPS in degrees and decimal minutes, converted). Keyed by the
# cast's start time so they apply to those exact files and nothing else.
EXAMPLE_POSITIONS = {
    "May 15 2026 09:25:45": (47.316683, -122.473050),   # Station 11  4719.001 N 12228.383 W
    "May 15 2026 10:45:48": (47.356100, -122.404350),   # Station 12  4721.366 N 12224.261 W
    "May 15 2026 12:46:01": (47.338617, -122.543950),   # Station 15  4720.317 N 12232.637 W
    "May 15 2026 13:13:28": (47.393233, -122.537217),   # Station 16  4723.594 N 12232.233 W
    "May 15 2026 13:40:34": (47.431667, -122.525000),   # Station 17  4725.900 N 12231.500 W
}


def station_position(df, meta):
    """Best available position for a cast, as (lat, lon) or (None, None).

    A Sea-Bird CTD has no GPS of its own. Position arrives from the ship's GPS
    through the deck unit's NMEA input, or is typed into Seasave before the
    cast — so an instrument run self-contained has none at all. Where the deck
    unit was set to append position to every scan there are latitude and
    longitude columns as well, and their median is a fair cast position."""
    lat, lon = meta.get("lat"), meta.get("lon")
    if lat is not None and lon is not None:
        return lat, lon
    la = _col_by_name(df, "latitude")
    lo = _col_by_name(df, "longitude")
    if la is not None and lo is not None:
        try:
            v1, v2 = float(df[la].median()), float(df[lo].median())
            if -90 <= v1 <= 90 and -180 <= v2 <= 180:
                return v1, v2
        except (TypeError, ValueError):
            pass
    if meta.get("time") in EXAMPLE_POSITIONS:
        return EXAMPLE_POSITIONS[meta["time"]]
    return None, None


def show_inline(fig):
    """Display a figure in the notebook without trusting the front end to run
    the plotly.js <script> before the plot script.

    In Colab, a figure shown after ipywidgets in the same cell can come up as
    an empty box the size of the plot: on that path the library tag is not
    waited for, Plotly.newPlot runs first and throws "Plotly is not defined"
    (googlecolab/colabtools#5889). fig.show() fails the same way there. So the
    library tag is left out, and the plot script waits for window.Plotly,
    loading it once per output frame if needed. If the CDN cannot be reached
    the box says so instead of staying blank."""
    import json
    from IPython.display import display, HTML
    try:
        from plotly.offline import get_plotlyjs_version
        cdn = f"https://cdn.plot.ly/plotly-{get_plotlyjs_version()}.min.js"
    except Exception:
        cdn = "https://cdn.plot.ly/plotly-2.35.2.min.js"
    html = fig.to_html(full_html=False, include_plotlyjs=False,
                       post_script=FIGURE_SCRIPT)
    m = re.search(r'id="([^"]+)" class="plotly-graph-div"', html)
    k = html.find("Plotly.newPlot")
    i0 = html.rfind("<script", 0, k)
    i1 = html.find(">", i0) + 1
    i2 = html.rfind("</script>")
    if not m or k < 0 or i0 < 0 or i2 < i1:
        # Unexpected markup from this plotly version: fall back to the plain form.
        display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn",
                                 post_script=FIGURE_SCRIPT)))
        return
    loader = (
        '<script type="text/javascript">(function () {\n'
        'function run() {' + html[i1:i2] + '}\n'
        'if (window.Plotly) { run(); return; }\n'
        'if (!window.__plotlyReady) {\n'
        '  window.__plotlyReady = new Promise(function (res, rej) {\n'
        '    var s = document.createElement("script"); s.src = ' + json.dumps(cdn) + ';\n'
        '    s.onload = res; s.onerror = rej; document.head.appendChild(s);\n'
        '  });\n'
        '}\n'
        'window.__plotlyReady.then(run, function () {\n'
        '  var gd = document.getElementById(' + json.dumps(m.group(1)) + ');\n'
        '  if (gd) { gd.textContent = "plotly.js could not be loaded from ' + cdn + '"; }\n'
        '});\n'
        '})();</script>')
    display(HTML(html[:i0] + loader + html[i2 + len("</script>"):]))


def station_map(positions=None, title=None, connect=False):
    """Station positions on an OpenStreetMap basemap, coloured to match the
    profile graphs. Returns None when nothing has coordinates.

    positions: {label: (lat, lon)}. Needs internet for the map tiles, so the
    offline export will show markers on a blank background."""
    pts = positions if positions is not None else {
        k: (m.get("lat"), m.get("lon")) for k, m in STATION_META.items()}
    pts = {k: v for k, v in pts.items()
           if v and v[0] is not None and v[1] is not None}
    if not pts:
        return None

    labels = list(pts)
    lats = [pts[k][0] for k in labels]
    lons = [pts[k][1] for k in labels]
    mid_lat = sum(lats) / len(lats)
    # Longitude degrees shrink with latitude, so compare spans in like units.
    span = max(max(lats) - min(lats),
               (max(lons) - min(lons)) * float(np.cos(np.radians(mid_lat))))
    zoom = next(z for lim, z in [(0.02, 12), (0.05, 11), (0.2, 10), (0.5, 9),
                                 (1, 8), (5, 6), (20, 4), (60, 3), (1e9, 1)]
                if span < lim)

    # plotly 6 renamed the Mapbox traces to Map and moved to MapLibre. Colab is
    # still on 5.x, which has only the old names, so support both.
    Trace = getattr(go, "Scattermap", None) or go.Scattermapbox
    map_key = "map" if hasattr(go, "Scattermap") else "mapbox"

    traces = []
    if connect and len(labels) > 1:
        traces.append(Trace(
            lat=lats, lon=lons, mode="lines", name="transect",
            line=dict(width=2, color="#555"), hoverinfo="skip", showlegend=False))
    for k in labels:
        la, lo = pts[k]
        traces.append(Trace(
            lat=[la], lon=[lo], mode="markers+text", name=k,
            marker=dict(size=13, color=STATION_COLORS.get(k, "#1f77b4")),
            text=[k], textposition="top right",
            textfont=dict(size=12),
            hovertemplate=f"<b>{k}</b><br>%{{lat:.4f}}, %{{lon:.4f}}<extra></extra>"))

    head = title or (f"{LOCATION}: stations" if LOCATION else "Stations")
    fig = go.Figure(traces)
    fig.update_layout(
        title=dict(text=head, x=0.5, xanchor="center", font=dict(size=16)),
        width=760, height=620, margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(title="Station"),
        **{map_key: dict(style="open-street-map",
                         center=dict(lat=mid_lat, lon=sum(lons) / len(lons)),
                         zoom=zoom)},
    )
    return fig


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres."""
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * r * np.arcsin(np.sqrt(a))


def transect_distances(order, positions):
    """Cumulative along-track distance in km for stations in the given order.
    Returns (labels, distances) for those that have a position."""
    have = [s for s in order
            if positions.get(s) and positions[s][0] is not None
            and positions[s][1] is not None]
    if len(have) < 2:
        return have, [0.0] * len(have)
    dist = [0.0]
    for a, b in zip(have, have[1:]):
        la1, lo1 = positions[a]
        la2, lo2 = positions[b]
        dist.append(dist[-1] + haversine_km(la1, lo1, la2, lo2))
    return have, dist


def _depth_ticks(lo, hi, n=8):
    """Round tick positions between lo and hi (never above the surface), so the
    headroom kept for the station labels does not show a negative depth tick."""
    raw = max(hi - lo, 1e-9) / n
    mag = 10 ** np.floor(np.log10(raw))
    step = next(m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= raw)
    # The surface reading is rarely exactly 0 m; start at 0 when it is close.
    lo = max(lo, 0.0)
    first = 0.0 if lo <= step * 0.05 else np.ceil(lo / step) * step
    return list(np.arange(first, hi + step * 0.01, step))


def seafloor_points(labels, dist, bottoms, midpoints=None):
    """Merge typed seafloor points into the cast bottoms.

    midpoints maps a station label to [(km, depth, towards), ...]: a depth read
    off a chart, measured from that station towards the neighbour named in
    'towards' (None means the next station in order). A point stays valid if
    the pair is still adjacent whichever way round it now sits. 'towards' can
    also be "<before>" or "<after>", a point out past the first or the last
    station: it counts only while its station is still at that end, and it
    widens the section that way. Returns the sorted (x, depth) list, the
    notes for points that were left out (and for any extension), how many
    were used, and the x extent (xmin, xmax) the section should cover."""
    pts = list(zip(dist, bottoms))
    notes, used = [], 0
    index = {lab: i for i, lab in enumerate(labels)}
    last = len(labels) - 1
    xmin, xmax = (dist[0], dist[last]) if labels else (0.0, 0.0)
    for lab, entries in (midpoints or {}).items():
        i = index.get(lab)
        if i is None:
            if entries:
                notes.append(f"{lab}: not on the transect, its seafloor points are not used")
            continue
        for entry in entries:
            d, z = entry[0], entry[1]
            to = entry[2] if len(entry) > 2 else None
            if d is None or z is None:
                notes.append(f"{lab}: a seafloor point with a box left empty is not used")
                continue
            try:
                d, z = float(d), float(z)
            except (TypeError, ValueError):
                continue
            if to in ("<before>", "<after>"):
                # out past an end of the line: only while this station is that end
                before = to == "<before>"
                if i != (0 if before else last):
                    notes.append(f"{lab}: seafloor point {'before' if before else 'beyond'} the line "
                                 f"skipped, the station is no longer at that end")
                elif not d > 0:
                    notes.append(f"{lab}: seafloor point at {d:g} km skipped, it must be more than "
                                 f"0 km from the station")
                elif not z > 0:
                    notes.append(f"{lab}: seafloor point at {d:g} km skipped, depth {z:g} m must be "
                                 f"more than 0")
                else:
                    x = dist[0] - d if before else dist[last] + d
                    pts.append((x, z))
                    xmin, xmax = min(xmin, x), max(xmax, x)
                    used += 1
                continue
            j = index.get(to) if to else i + 1
            if j is None or abs(j - i) != 1:
                notes.append(f"{lab}: seafloor point at {d:g} km skipped, {to or 'the next station'} "
                             f"is no longer next to it")
                continue
            if j >= len(labels):
                notes.append(f"{lab}: seafloor points after the last station are not used")
                continue
            seg = abs(dist[j] - dist[i])
            if not d > 0:
                notes.append(f"{lab}: seafloor point at {d:g} km skipped, it must be more than "
                             f"0 km from the station")
            elif not d < seg:
                notes.append(f"{lab}: seafloor point at {d:g} km skipped, {labels[j]} is only "
                             f"{seg:.2f} km away")
            elif not z > 0:
                notes.append(f"{lab}: seafloor point at {d:g} km skipped, depth {z:g} m must be "
                             f"more than 0")
            else:
                pts.append((dist[i] + d if j > i else dist[i] - d, z))
                used += 1
    pts.sort()
    # past an end station the field repeats that station's column, so say so
    if labels and xmin < dist[0]:
        notes.append(f"extended {dist[0] - xmin:.2f} km before {labels[0]}, "
                     f"colours there repeat that station")
    if labels and xmax > dist[last]:
        notes.append(f"extended {xmax - dist[last]:.2f} km beyond {labels[last]}, "
                     f"colours there repeat that station")
    return pts, notes, used, xmin, xmax


def build_section(stations, series, positions, order=None, depth_min=None,
                  depth_max=None, grid=(240, 200), n_contours=0,
                  colorscale_name=None, names=None, midpoints=None,
                  colorscale_stops=None, color_range=None, colorbar_name=False):
    """Vertical section: distance along the transect against depth, coloured by
    one variable.

    Interpolation is done in two passes — each cast onto a common depth grid,
    then across stations at every depth. Below a cast's deepest reading its
    last value is carried down to the seafloor line between stations, so the
    field meets the black polygon instead of stopping in a white wedge. That
    part is inferred, like everything between the stations.

    Interpolating between stations invents structure that was never measured.
    Station positions are drawn on top so it is obvious where the real data is.
    names maps a station label to the text shown above its marker (defaults to
    the label). midpoints are extra seafloor points read off a chart, see
    seafloor_points(); they shape the black polygon and extend the depth axis,
    and the colour between stations is stretched down to meet them, so what
    lies below a cast's deepest reading is inferred. Returns None if fewer
    than two stations have both a position and data.

    colorscale_stops is a Plotly colorscale (from read_palette) used instead of
    the variable's own map; color_range a (low, high) used instead of its fixed
    range; colorbar_name puts "Temperature (\u00b0C)" on the colour bar rather
    than just the unit."""
    if not STATION_COLORS:
        assign_station_styles([s for s, _ in stations])
    frames = dict(stations)
    order = order or [s for s, _ in stations]
    labels, dist = transect_distances([s for s in order if s in frames], positions)
    labels = [s for s in labels if s in frames]
    if len(labels) < 2:
        return None

    # each cast onto a shared depth grid
    profiles, bottoms = [], []
    for lab in labels:
        df = frames[lab]
        xcol = _col_by_name(df, series["col"])
        ycol, _lab = find_channel(df, DEPTH_CANDIDATES)
        if xcol is None or ycol is None:
            return None
        sub = df[[xcol, ycol]].dropna().sort_values(ycol)
        if depth_min is not None:
            sub = sub[sub[ycol] >= depth_min]
        if depth_max is not None:
            sub = sub[sub[ycol] <= depth_max]
        if sub.empty:
            return None
        profiles.append((sub[ycol].to_numpy(float), sub[xcol].to_numpy(float)))
        bottoms.append(float(sub[ycol].max()))

    # The seafloor line: each cast's deepest reading plus the typed points. A
    # point before the first station or beyond the last widens the section,
    # so the x extent comes back with the points.
    floor_pts, _notes, _used, xmin, xmax = seafloor_points(labels, dist, bottoms, midpoints)
    deepest_floor = max(z for _x, z in floor_pts)

    top = depth_min if depth_min is not None else min(p[0].min() for p in profiles)
    bot = depth_max if depth_max is not None else max(max(bottoms), deepest_floor)
    nx, ny = grid
    xs = np.linspace(xmin, xmax, nx)
    ys = np.linspace(max(0.0, top), bot, ny)

    # values at each station on the common depth grid; the surface value is
    # held up to 0 m and the deepest value down to the seafloor line
    columns = []
    for d, v in profiles:
        columns.append(np.interp(ys, d, v, left=v[0], right=v[-1]))
    columns = np.array(columns)                      # station x depth

    # horizontal pass: interpolate across stations at every depth. Past the
    # end stations each one's own column is held constant, so an extension
    # is coloured like the station it reaches out from.
    z = np.full((ny, nx), np.nan)
    for j in range(ny):
        row = columns[:, j]
        ok = ~np.isnan(row)
        if ok.sum() >= 2:
            z[j] = np.interp(xs, np.array(dist)[ok], row[ok],
                             left=row[ok][0], right=row[ok][-1])
        elif ok.sum() == 1:
            z[j] = np.where(np.isclose(xs, np.array(dist)[ok][0]), row[ok][0], np.nan)

    # The field is left intact below the seafloor line and the black polygon
    # covers it, so there is no gap along the edge. The polygon is sampled on
    # the grid plus the typed points themselves, so it reaches each one
    # exactly, and is clipped to the depth window so it never leaves the axis.
    px = np.union1d(xs, [x for x, _z in floor_pts])
    floor = np.interp(px, [x for x, _z in floor_pts], [z for _x, z in floor_pts])

    scale = (colorscale_stops if colorscale_stops
             else colorscale(colorscale_name) if colorscale_name
             else variable_colorscale(series["name"]))
    unit = series["label"][len(series["name"]):].strip(" ()") if "(" in series["label"] else ""
    unit_text = pretty_units(unit)

    # Colour is fixed per variable (VARIABLE_RANGES) so the same colour means
    # the same value on every section. A palette with levels overrides that
    # through color_range, and the data's own extent is the fallback. The
    # colour bar ticks start at the low end and step by a round number, so
    # the labels read 6, 8, 10 rather than 6.37, 8.91.
    fixed = variable_range(series["name"], unit)
    tick = None
    if color_range:
        lo_z, hi_z = sorted(float(v) for v in color_range)
    elif fixed:
        lo_z, hi_z, tick = fixed
    else:
        lo_z, hi_z = float(np.nanmin(z)), float(np.nanmax(z))
    if not hi_z > lo_z:                 # a flat field still needs a range to draw
        lo_z, hi_z = lo_z - 0.5, hi_z + 0.5
    if tick is None:
        tick = nice_step(hi_z - lo_z, 7)

    # n_contours 0 leaves it a smooth gradient; anything else bands the colour
    # range into that many steps, which makes layering obvious at a glance.
    # Contour lines carry their value in a gap cut into the line. Plotly draws
    # the lines in both modes (showlines only matters for "fill").
    label_style = dict(showlabels=True, labelfont=dict(size=9, color="#111"))
    line = dict(width=0.5, color="rgba(0,0,0,0.35)")
    if n_contours and n_contours > 0:
        contours = dict(coloring="fill", showlines=True, start=lo_z, end=hi_z,
                        size=max((hi_z - lo_z) / max(n_contours, 1), 1e-9), **label_style)
        # ncontours must be >= 1 in plotly, so it is left out entirely for the
        # smooth case rather than passed as 0.
        extra = dict(ncontours=int(n_contours), line=line)
    else:
        contours = dict(coloring="heatmap", **label_style)
        extra = dict(line=line)

    traces = [go.Contour(
        x=xs, y=ys, z=z, colorscale=scale, contours=contours, connectgaps=False,
        zmin=lo_z, zmax=hi_z, zauto=False,
        colorbar=dict(title=dict(text=(label_with_units(series["name"], unit) if colorbar_name
                                       else unit_text or series["name"]), side="right"),
                      thickness=14, len=0.9, tick0=lo_z, dtick=tick, tickformat=".2f"),
        hovertemplate=("%{x:.2f} km<br>%{y:.1f} m<br>"
                       f"{series['name']}: %{{z:.3f}} {unit_text}<extra></extra>"),
        **extra,
    )]

    # Headroom above the shallowest reading for the station markers: a triangle
    # pointing down at the field, with the station name centred above it.
    surface = max(0.0, top)
    span = max(bot - surface, 1e-9)
    head_room = span * 0.13
    y_marker = surface - head_room * 0.3

    # The seafloor as a solid black polygon: the floor line implied by the casts,
    # closed along the bottom of the axis. Drawn after the field so it covers it.
    axis_bottom = bot + span * 0.02
    traces.append(go.Scatter(
        x=list(px) + [float(px[-1]), float(px[0])],
        y=list(np.clip(floor, surface, bot)) + [axis_bottom, axis_bottom],
        fill="toself", fillcolor="#000000", mode="lines",
        line=dict(width=0, color="#000000"),
        hoverinfo="skip", showlegend=False, name="seafloor"))

    # One marker per station, name above it. The markers are not hoverable;
    # the field underneath is, so a hover anywhere still reads the value.
    for lab, d in zip(labels, dist):
        traces.append(go.Scatter(
            x=[d], y=[y_marker], mode="markers+text", name=lab,
            marker=dict(symbol="triangle-down", size=13,
                        color=STATION_COLORS.get(lab, "#333"),
                        line=dict(color="white", width=1)),
            text=[(names or {}).get(lab) or lab], textposition="top center",
            textfont=dict(size=11),
            cliponaxis=False, hoverinfo="skip", showlegend=False))

    head = f"{series['name']} section"
    if LOCATION:
        head = f"{LOCATION}: {head}"
    fig = go.Figure(traces)
    # No grid across the field — it competes with the contours. Small outward
    # ticks stay, so the numbers still read against the axis.
    axis_style = dict(showgrid=False, zeroline=False, showline=True,
                      linecolor="#888", ticks="outside", ticklen=4,
                      tickcolor="#888", tickfont=dict(size=10),
                      fixedrange=True)
    fig.update_layout(
        title=dict(text=head if SHOW_TITLES else "", x=0.5, xanchor="center",
                   font=dict(size=16)),
        xaxis=dict(title=dict(text="Distance along transect (km)",
                              font=dict(size=AXIS_FONT), standoff=8),
                   range=[xmin, xmax], **axis_style),
        yaxis=dict(title=dict(text="Depth (m)", font=dict(size=AXIS_FONT), standoff=8),
                   range=[axis_bottom, surface - head_room],
                   tickvals=_depth_ticks(surface, bot), **axis_style),
        width=860, height=560, **_graph_theme(),
        margin=dict(l=70, r=30, t=70 if SHOW_TITLES else 30, b=60), showlegend=False,
        # Static by design: hover for values and nothing else. No drag or wheel
        # zoom, no pan, no box or lasso select, no legend to click. The toolbar
        # keeps only the PNG button (FIGURE_SCRIPT leaves reset out as well).
        dragmode=False, hovermode="closest",
        legend=dict(itemclick=False, itemdoubleclick=False),
        modebar=dict(remove=["zoom2d", "pan2d", "select2d", "lasso2d", "zoomIn2d",
                             "zoomOut2d", "autoScale2d", "resetScale2d"]),
    )
    return fig


# Injected into every exported figure. Two behaviours plotly does not give for
# free: box and lasso select act like clicking the legend, and the PNG button
# drops hidden stations from the legend instead of showing them greyed out.
FIGURE_SCRIPT = """
(function () {
  var gd = document.getElementById('{plot_id}');
  if (!gd || !window.Plotly) { return; }

  // A figure made with dragmode=False is static by design (the transect):
  // hover only. It gets the PNG button and nothing else from this script.
  var isStatic = !!(gd._fullLayout && gd._fullLayout.dragmode === false);

  // What the figure looked like when it was made: trace visibility, the
  // legend, and the view. The reset button puts all of it back.
  var initial = {
    layout: JSON.parse(JSON.stringify(gd.layout)),
    dragmode: gd._fullLayout && gd._fullLayout.dragmode,
    visible: gd.data.map(function (t) { return t.visible === undefined ? true : t.visible; }),
    showlegend: gd.data.map(function (t) { return t.showlegend === undefined ? true : t.showlegend; })
  };

  function legendTraces() {
    var idx = [];
    gd.data.forEach(function (t, i) { if (t.showlegend !== false) { idx.push(i); } });
    return idx;
  }

  // Box and lasso select do exactly what clicking the legend does: every
  // station outside the selection is switched off, greyed in the legend and
  // gone from the plot, and a legend click brings it back. Plotly's own
  // dimming of unselected points is cleared at the same time, otherwise a
  // station brought back later would still draw greyed out.
  gd.on('plotly_selected', function (ev) {
    if (isStatic || !ev || !ev.points || !ev.points.length) { return; }
    var picked = {};
    ev.points.forEach(function (p) { picked[p.curveNumber] = true; });
    var idx = legendTraces();
    var vis = idx.map(function (i) { return picked[i] ? true : 'legendonly'; });
    // Plotly records the selection after firing this event, so tidy up on
    // the next tick. The dashed selection box has to go first: while it
    // exists, clearing the dimmed points is undone straight away.
    setTimeout(function () {
      Plotly.relayout(gd, { selections: null }).then(function () {
        return Plotly.restyle(gd, { visible: vis, selectedpoints: null }, idx);
      });
    }, 0);
  });

  // Double-click while in select mode clears the selection: show everything.
  gd.on('plotly_deselect', function () {
    if (isStatic) { return; }
    var idx = legendTraces();
    Plotly.restyle(gd, { visible: true, selectedpoints: null }, idx);
  });

  // A switched-off station is already absent from the plot; for the PNG it
  // should be absent from the legend too. Drop it from the legend, snapshot,
  // then put it back. Only the legend is touched, so the map is not redrawn.
  function hidden() {
    var idx = [];
    gd.data.forEach(function (t, i) { if (t.visible === 'legendonly') { idx.push(i); } });
    return idx;
  }

  function download() {
    var name = 'chart';
    try {
      var t = gd.layout.title;
      name = (typeof t === 'string' ? t : (t && t.text) || 'chart')
             .replace(/[^A-Za-z0-9]+/g, '_').replace(/^_|_$/g, '') || 'chart';
    } catch (e) {}
    return Plotly.downloadImage(gd, { format: 'png', scale: 3, filename: name });
  }

  var button = {
    name: 'downloadVisible',
    title: isStatic ? 'Download PNG' : 'Download PNG (hidden stations left out)',
    icon: Plotly.Icons.camera,
    click: function () {
      var idx = hidden();
      if (!idx.length) { download(); return; }
      var restore = function () { Plotly.restyle(gd, { showlegend: true }, idx); };
      // If the snapshot never settles (a map that cannot render offscreen),
      // still give the legend back after a while.
      var settled = false;
      var done = function () { if (!settled) { settled = true; restore(); } };
      setTimeout(done, 20000);
      Plotly.restyle(gd, { showlegend: false }, idx).then(download).then(done, done);
    }
  };

  function reset() {
    var upd = { selections: null };
    if (initial.dragmode) { upd.dragmode = initial.dragmode; }
    var lay = initial.layout;
    Object.keys(gd._fullLayout).forEach(function (k) {
      if (/^[xy]axisd*$/.test(k)) {
        var ax = lay[k] || {};
        if (ax.range && !ax.autorange) {
          upd[k + '.range'] = ax.range.slice();
          upd[k + '.autorange'] = false;
        } else {
          upd[k + '.autorange'] = true;
        }
      }
    });
    Object.keys(lay).forEach(function (k) {
      if (/^(mapbox|map)d*$/.test(k)) {
        ['center', 'zoom', 'bearing', 'pitch'].forEach(function (p) {
          if (lay[k][p] !== undefined) { upd[k + '.' + p] = JSON.parse(JSON.stringify(lay[k][p])); }
        });
      }
    });
    Plotly.restyle(gd, { visible: initial.visible, showlegend: initial.showlegend, selectedpoints: null })
          .then(function () { return Plotly.relayout(gd, upd); });
  }

  var resetButton = {
    name: 'resetChart',
    title: 'Reset chart (original view, all stations)',
    icon: Plotly.Icons.undo,
    click: reset
  };

  // Swap the toolbar without redrawing anything: the toolbar is rebuilt from
  // the graph's config whenever a modebar layout attribute changes, and this
  // never touches the map, whose style may still be loading.
  try {
    gd._context.displaylogo = false;
    gd._context.modeBarButtonsToAdd = isStatic ? [button] : [button, resetButton];
    // Keep whatever the figure already asked to have removed.
    var already = (gd.layout.modebar && gd.layout.modebar.remove) || [];
    if (typeof already === 'string') { already = [already]; }
    Plotly.relayout(gd, { 'modebar.remove': already.concat(['toImage']) });
  } catch (e) { /* leave the default toolbar rather than breaking the plot */ }
})();
"""


def write_combined_html(figs, path, title="CTD Profiles", inline=False, show_heading=True):
    """All figures in one HTML file.

    inline=False links the plotting library from the web: ~30 KB, opens at once,
    emails and uploads without trouble, but needs a connection to draw.
    inline=True bakes the library in: several MB, works with no internet. Keep
    that one for presenting somewhere without wifi — a file that large is slow
    to open and easy to truncate in transit, so it is a poor default."""
    parts = [f.to_html(full_html=False,
                       include_plotlyjs=("inline" if inline else "cdn") if i == 0 else False,
                       post_script=FIGURE_SCRIPT)
             for i, (_, f) in enumerate(figs)]
    html = (
        '<!doctype html><html><head><meta charset="utf-8">'
        f"<title>{title}</title><style>"
        "body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif;margin:24px;"
        "background:#fff;color:#111}h1{font-size:20px;font-weight:600}"
        ".grid{display:flex;flex-wrap:wrap;gap:16px}"
        # never let a figure be squeezed to nothing by the flex container
        ".grid>div{flex:0 0 auto}</style></head><body>"
        + (f"<h1>{title}</h1>" if show_heading else "")
        + f"<div class=\"grid\">{''.join(parts)}</div></body></html>"
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


def export_figures(figs, tag="", want_png=True, stem="CTD_profiles"):
    """Write the interactive HTML (+ optional PNGs) and return a status string."""
    os.makedirs(os.path.join(OUT_DIR, "png"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, "single_graphs"), exist_ok=True)
    base = f"{LOCATION} — CTD Profiles" if LOCATION else "CTD Profiles"
    html_path = os.path.join(OUT_DIR, f"{stem}.html")
    offline_path = os.path.join(OUT_DIR, f"{stem}_offline.html")
    write_combined_html(figs, html_path, title=f"{base}{tag}", inline=False)
    write_combined_html(figs, offline_path, title=f"{base}{tag}", inline=True)

    # One file per variable, so a single graph can be shared or embedded on its
    # own without anyone having to edit HTML. No heading: the figure carries its
    # own title, and a bare graph embeds more cleanly in someone else's page.
    for name, fig in figs:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
        one = f"{LOCATION}: Depth vs {name}" if LOCATION else f"Depth vs {name}"
        write_combined_html([(name, fig)],
                            os.path.join(OUT_DIR, "single_graphs", f"{safe}.html"),
                            title=one, inline=False, show_heading=False)

    msg = (f"{html_path} ({os.path.getsize(html_path)/1e3:.0f} KB)"
           f" + offline copy ({os.path.getsize(offline_path)/1e6:.1f} MB)"
           f" + {len(figs)} single graphs")
    if want_png and EXPORT_PNG:
        try:
            for name, fig in figs:
                safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
                fig.write_image(os.path.join(OUT_DIR, "png", f"{safe}.png"), scale=3)
            msg += f" · {len(figs)} PNGs"
        except Exception as e:
            msg += f" · PNG skipped ({type(e).__name__})"
    return msg


print("Setup complete.  markers:", SHOW_MARKERS, "· x-padding:", f"{X_PAD_FRAC:.0%}")

In [ ]:
#@title 1 · Survey location and files
#@markdown **Survey location** — the name of the overall area you sampled.
#@markdown It becomes the title of every graph, for example
#@markdown `Quartermaster Harbor: Depth vs Temperature`. Leave it blank to get
#@markdown just `Depth vs Temperature`.
survey_location = "" #@param {type:"string"}
#@markdown ---
#@markdown **Keep the downcast only.** The instrument records on the way down and
#@markdown again on the way back up. An unprocessed file holds both, so the line
#@markdown retraces itself. Ticked, only the downward half is kept. Files that
#@markdown were already processed are left alone.
#@markdown
#@markdown This fixes the **shape, not the numbers** — see Raw casts in the
#@markdown [README](https://github.com/jimothy-dev/CTD_Grapher_v2#raw-casts).
downcast_only_raw = True #@param {type:"boolean"}
#@markdown ---
#@markdown Run this cell and pick your `.cnv` files when the upload button appears.
#@markdown Picked the wrong file? Run this cell again and re-upload — only the
#@markdown newest copy of each station is used.

import ipywidgets as widgets
from IPython.display import display

LOCATION = survey_location.strip()
raw = load_files()

stations = []
# Sort on the cleaned station name, never the filename — a ' (1)' left by a
# re-upload would otherwise reorder stations and shuffle every colour.
for fn in sorted(raw, key=lambda f: natkey(station_label(f))):
    try:
        df, proc = parse_cnv(raw[fn], fn)
    except ValueError as e:
        print(f"  SKIPPED  {e}")
        continue
    if df.empty:
        print(f"  SKIPPED  {fn}: no data rows after *END*")
        continue

    label = station_label(fn)
    dcol, dlab = find_channel(df, DEPTH_CANDIDATES)

    note = ""
    if "loopedit" not in proc:
        if downcast_only_raw:
            df, dropped, time_series = downcast_only(df, dcol)
            if time_series:
                note = ("  ← looks like a time series at one depth rather than a "
                        "cast, so nothing was cut")
            else:
                # Not "loopedit" — that is a Sea-Bird step this notebook does not run.
                proc = proc | {"downcast cut (this notebook)"}
                note = f"  ← raw cast, kept the downcast ({dropped:,} rows dropped)"
        else:
            note = "  ← raw cast, may double back on itself"

    # A tidy-looking profile should not imply the values were corrected.
    missing = [s for s in ("align", "celltm") if not any(s in p for p in proc)]
    if missing and "derive" in proc:
        note += ("\n      NOTE  this file's header shows no correction steps, so the "
                 "values may be off. See Raw casts in the README.")

    stations.append((label, df))
    STATION_META[label] = parse_header_meta(raw[fn])
    drange = f"{df[dcol].min():.1f}–{df[dcol].max():.1f}" if dcol else "no depth column"
    print(f"  {label:<28} {len(df):>6,} rows   {drange:<16} "
          f"processing: {', '.join(sorted(proc)) or 'none'}{note}")

if not stations:
    raise SystemExit("No readable .cnv files found.")

DEEPEST = max(df[find_channel(df, DEPTH_CANDIDATES)[0]].max() for _, df in stations)
assign_station_styles([lab for lab, _ in stations])

print(f"\n{len(stations)} station(s) loaded · deepest reading {DEEPEST:.1f} m")
print(f"Location: {LOCATION or '(none set)'}")
print("\nColour key — reuse these in any other chart of the same stations:")
for lab, col in STATION_COLORS.items():
    print(f"  {col}   {lab}")


# ─────────── choose what to plot ───────────
RECOGNISED, EXTRAS, Y_DEFAULT = available_channels(stations)


def _row(chan, ticked):
    box = widgets.Checkbox(value=ticked, description=chan["name"], indent=False,
                           layout=widgets.Layout(width="250px"))
    txt = widgets.Text(value=chan["label"], layout=widgets.Layout(width="270px"))
    return box, txt, chan


# The classic variables start ticked. Derived and engineering channels are
# recognised but unticked, and anything unrecognised is offered unticked too, so
# nothing is hidden and the default output stays the usual set of graphs.
CHANNEL_ROWS = ([_row(c, c.get("on", True)) for c in RECOGNISED]
                + [_row(c, False) for c in EXTRAS])

Y_CHOICES = {c["name"]: c for c in [Y_DEFAULT] + RECOGNISED + EXTRAS}
Y_PICK = widgets.Dropdown(options=list(Y_CHOICES), value=Y_DEFAULT["name"],
                          description="Y axis:", style={"description_width": "62px"},
                          layout=widgets.Layout(width="290px"))
Y_LABEL = widgets.Text(value=Y_DEFAULT["label"], description="Y label:",
                       style={"description_width": "62px"},
                       layout=widgets.Layout(width="290px"))
Y_INVERT = widgets.Checkbox(value=True, description="Invert Y axis (largest at the bottom)",
                            indent=False, layout=widgets.Layout(width="360px"))
Y_PICK.observe(lambda _c: setattr(Y_LABEL, "value", Y_CHOICES[Y_PICK.value]["label"]),
               names="value")


def selected_series():
    """Ticked channels, with whatever axis label sits in the box beside each."""
    return [{"name": ch["name"], "col": ch["col"], "label": txt.value}
            for box, txt, ch in CHANNEL_ROWS if box.value]


def selected_y():
    # is_depth only decides whether the axis is anchored to the requested window
    # so 0 m stays visible. The depth window itself filters readings and applies
    # whatever is on the axes. invert is purely the tick box.
    ch = Y_CHOICES[Y_PICK.value]
    return {"name": ch["name"], "col": ch["col"], "label": Y_LABEL.value,
            "is_depth": bool(ch.get("is_depth", False)),
            "invert": bool(Y_INVERT.value)}


# Live, unlike the form field above: the Plot cell reads this box each time it
# runs, so the name can be changed without re-running anything else.
LOCATION_BOX = widgets.Text(value=LOCATION, description="Location:",
                            placeholder="e.g. Quartermaster Harbor",
                            style={"description_width": "70px"},
                            layout=widgets.Layout(width="420px"))
display(widgets.HTML("<b>Survey location</b> &mdash; used in every graph title. "
                     "Change it any time and re-run only the Plot cell."))
display(LOCATION_BOX)

display(widgets.HTML(
    "<b>Plot these</b> &mdash; tick what you want a graph of. The box beside each "
    "is its axis label, units included; edit it if you like.<br>"
    "Recognised variables are already ticked. The rest are other columns your "
    "files happen to contain, in case you want them."))
display(widgets.VBox([widgets.HBox([b, t]) for b, t, _ in CHANNEL_ROWS]))
display(widgets.HTML(
    "<b>Y axis</b> &mdash; depth unless you change it. Pick something else and the "
    "graphs become ordinary scatter plots titled <i>Salinity vs Temperature</i> "
    "and so on. The depth window in the next cell still applies."))
display(widgets.HBox([Y_PICK, Y_LABEL]))
display(Y_INVERT)


print("\nNow run cell 2 for the graphs. Positions and the transect are in cell 3.")


In [ ]:
#@title 2 · Draw the graphs
#@markdown **Which part of the water column to show**, in metres below the surface.
#@markdown Leave both blank to show the whole cast, top to bottom.
#@markdown
#@markdown To look at just the surface layer, put `0` and `20` — that shows
#@markdown everything between 0 and 20 metres deep. Change the numbers and run
#@markdown this cell again; your files stay loaded. Applies whatever is on the
#@markdown axes, since it filters readings by how deep they were taken.
depth_from_m = "" #@param {type:"string"}
depth_to_m   = "" #@param {type:"string"}
#@markdown ---
#@markdown **How the graphs look** — where the legend sits, a dark background,
#@markdown whether each graph carries a title, and whether the y label runs along
#@markdown the axis or sits upright above it.
legend = "right" #@param ["left", "right", "bottom"]
dark_graphs = False #@param {type:"boolean"}
show_titles = True #@param {type:"boolean"}
y_label = "along the axis" #@param ["along the axis", "upright at top"]
#@markdown ---
#@markdown **Extra graphs**, any variable against any other, separated by semicolons,
#@markdown e.g. `Temperature vs Salinity; Fluorescence vs Depth` (the first name is X,
#@markdown the second Y; Depth is the default Y for the main graphs).
extra_graphs = "" #@param {type:"string"}

try:
    stations
except NameError:
    raise SystemExit("Run the Load cell first.")


def _num(s):
    s = str(s).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        print(f"  ignoring '{s}' — that is not a number")
        return None


# Read live, so the name and the ticked stations can change without re-running
# the Load cell.
LOCATION = LOCATION_BOX.value.strip()
y = selected_y()
series = [s for s in selected_series() if s["col"].lower() != y["col"].lower()]

if not series:
    raise SystemExit("Nothing ticked to plot. Tick at least one variable in the Load cell.")

dmin, dmax = _num(depth_from_m), _num(depth_to_m)
if dmin is not None and dmax is not None and dmin > dmax:
    dmin, dmax = dmax, dmin

# The look of the graphs: Setup settings, set here so build_figures sees them.
LEGEND_POS, DARK_GRAPHS, SHOW_TITLES = legend, dark_graphs, show_titles
Y_LABEL_TOP = y_label == "upright at top"

figs = build_figures(stations, series, y, dmin, dmax)
main_names = [n for n, _ in figs]

# Extra graphs from the box above, "X vs Y". Names are matched, ignoring case,
# against the channels the Load cell offers; a ticked one carries the label
# typed beside it, and so does the current Y axis. Y reads downward only when
# it is depth (or pressure standing in for depth), and then follows the
# invert tick box.
_by_name = {c["name"].lower(): c for c in Y_CHOICES.values()}
for _c in selected_series():
    _by_name[_c["name"].lower()] = _c
_by_name[y["name"].lower()] = y            # carries the live Y label box
_by_name.setdefault("depth", Y_DEFAULT)     # "Depth" still works when pressure stands in
for _pair in extra_graphs.split(";"):
    if not _pair.strip():
        continue
    _names = re.split(r"\s+vs\.?\s+", _pair.strip(), flags=re.I)
    if len(_names) != 2:
        print(f"  extra graph '{_pair.strip()}' skipped: write it as X vs Y")
        continue
    _xc, _yc = (_by_name.get(n.strip().lower()) for n in _names)
    _unknown = [n.strip() for n, c in zip(_names, (_xc, _yc)) if c is None]
    if _unknown:
        print(f"  extra graph '{_pair.strip()}' skipped: no channel called "
              + " or ".join(f"'{n}'" for n in _unknown))
        continue
    if str(_xc["col"]).lower() == str(_yc["col"]).lower():
        print(f"  extra graph '{_pair.strip()}' skipped: X and Y are the same channel")
        continue
    _y = {"name": _yc["name"], "col": _yc["col"], "label": _yc["label"],
          "is_depth": bool(_yc.get("is_depth", False)),
          "invert": bool(_yc.get("is_depth", False)) and bool(Y_INVERT.value)}
    for _n, _f in build_figures(stations, [_xc], _y, dmin, dmax):
        figs.append((f"{_y['name']} vs {_n}", _f))

if not figs:
    where = f" between {dmin:g} and {dmax:g} m" if (dmin or dmax) else ""
    print(f"Nothing to draw{where}. The deepest reading in this set is {DEEPEST:.1f} m.")
else:
    span = ""
    if dmin is not None or dmax is not None:
        lo = f"{dmin:g}" if dmin is not None else "0"
        hi = f"{dmax:g}" if dmax is not None else f"{DEEPEST:.0f}"
        span = f" · {lo}–{hi} m"
    if main_names:
        print(f"{y['name']} vs: " + ", ".join(main_names) + span)
    if len(figs) > len(main_names):
        print("Extra: " + ", ".join(n for n, _ in figs[len(main_names):]) + span)
    print("Saved:", export_figures(figs, tag=span))
    # Plain fig.show() with no clear_output(). Colab's plotly renderer attaches a
    # MutationObserver that purges the plot when its output element is rebuilt, so
    # clearing and redrawing the cell destroys the figures as they arrive.
    for _, f in figs:
        f.show(post_script=FIGURE_SCRIPT)

In [ ]:
#@title 3 · Station positions and transect
#@markdown Only needed for the station map and the transect: the comparison
#@markdown graphs in cell 2 work without any of this.
#@markdown
#@markdown **Positions** are filled in where the file knows them, or for the
#@markdown example casts; type in the rest. **Transect** — tick the stations on
#@markdown the line, drag them into the order they lie along it, edit the label
#@markdown shown above each one on the graph, and add any seafloor depths you
#@markdown know between stations. Then run cell 4.

try:
    stations
except NameError:
    raise SystemExit("Run the Load cell first.")

import json
import ipywidgets as widgets
from IPython.display import display, HTML

# ─────────── station positions ───────────
# A Sea-Bird CTD has no GPS. Position comes from the ship's GPS via the deck
# unit's NMEA input, or is typed into Seasave before the cast, so plenty of
# files have none. These boxes are prefilled where the file knows, and typed in
# where it does not.
def _pos_row(label, df, meta):
    lat, lon = station_position(df, meta)
    name = widgets.HTML(f"<div style='width:150px;padding-top:4px'>{label}</div>")
    la = widgets.Text(value="" if lat is None else f"{abs(lat):.5f}",
                      placeholder="latitude", layout=widgets.Layout(width="150px"))
    ns = widgets.Dropdown(options=["N", "S"], value="S" if (lat or 0) < 0 else "N",
                          layout=widgets.Layout(width="62px"))
    lo = widgets.Text(value="" if lon is None else f"{abs(lon):.5f}",
                      placeholder="longitude", layout=widgets.Layout(width="150px"))
    ew = widgets.Dropdown(options=["E", "W"], value="W" if (lon or 0) < 0 else "E",
                          layout=widgets.Layout(width="62px"))
    out = widgets.HTML(layout=widgets.Layout(width="430px"))
    row = (name, la, ns, lo, ew, out, label)

    def refresh(_change=None):
        # A hemisphere letter or minus typed into the box wins over the
        # dropdown, so pasted coordinates behave as written.
        v1, n1 = parse_coordinate(la.value, "lat")
        v2, n2 = parse_coordinate(lo.value, "lon")
        if v1 is not None and not re.search(r"[NSns\-]", la.value):
            v1 = abs(v1) * (-1 if ns.value == "S" else 1)
        if v2 is not None and not re.search(r"[EWew\-]", lo.value):
            v2 = abs(v2) * (-1 if ew.value == "W" else 1)
        bits = []
        for v, note, kind, raw in ((v1, n1, "lat", la.value), (v2, n2, "lon", lo.value)):
            if raw.strip() == "":
                continue
            bits.append(f"<span style='color:#0a0'>{format_coordinate(v, kind)}</span>"
                        if v is not None else
                        f"<span style='color:#b00'>{kind}: {note}</span>")
        out.value = ("<div style='padding-top:4px;font-family:monospace;font-size:11px'>"
                     + " &nbsp; ".join(bits) + "</div>")
        m = STATION_META.setdefault(label, {})
        m["lat"], m["lon"] = v1, v2

    for w in (la, lo, ns, ew):
        w.observe(refresh, names="value")
    refresh()
    return row


POSITION_ROWS = [_pos_row(lab, df, STATION_META.get(lab, {})) for lab, df in stations]


# One click sets the hemisphere on every row. A survey sits in one quadrant,
# so N/S and E/W should not need choosing station by station.
def _all_rows(index, value):
    for row in POSITION_ROWS:
        row[index].value = value


ALL_NS = widgets.ToggleButtons(options=["N", "S"], value=POSITION_ROWS[0][2].value,
                               description="All lat:", style={"description_width": "50px", "button_width": "34px"},
                               layout=widgets.Layout(width="160px"))
ALL_EW = widgets.ToggleButtons(options=["E", "W"], value=POSITION_ROWS[0][4].value,
                               description="All lon:", style={"description_width": "50px", "button_width": "34px"},
                               layout=widgets.Layout(width="160px"))
ALL_NS.observe(lambda c: _all_rows(2, c["new"]), names="value")
ALL_EW.observe(lambda c: _all_rows(4, c["new"]), names="value")


def apply_positions():
    """STATION_META is kept current by the boxes themselves; this just reports."""
    return {k: (m.get("lat"), m.get("lon")) for k, m in STATION_META.items()}


_have = sum(1 for r in POSITION_ROWS if r[1].value)
display(widgets.HTML(
    f"<b>Station positions</b> &mdash; {_have} of {len(POSITION_ROWS)} already known "
    f"(file headers, or the example casts). A Sea-Bird CTD has no GPS of its own, so a "
    f"cast run without a deck unit feeding it NMEA records no position; type those in "
    f"for a station map. The buttons set the hemisphere on every row at once."
    f"<br>Any usual format works &mdash; <code>47.4012</code>, "
    f"<code>47 24.072</code>, <code>47°24'04.3\"</code>, <code>4724.072</code> "
    f"&mdash; and the converted value is shown beside each box."))
display(widgets.HBox([ALL_NS, ALL_EW]))
display(widgets.VBox([widgets.HBox([n, la, ns, lo, ew, out])
                      for n, la, ns, lo, ew, out, _lab in POSITION_ROWS]))

# ─────────── stations on the transect ───────────
# A draggable list in plain HTML, since ipywidgets has no such thing. Every
# change (order, tick, label) is sent back through Colab's callback bridge and
# kept in TRANSECT_ORDER and TRANSECT_STATE, which survive re-running this
# cell and are what cell 4 reads. Unpositioned stations are listed too, so the
# order can be set before every position is typed in; they are left out of
# the transect until they have one.
_labels = [lab for lab, _ in stations]
TRANSECT_STATE = {k: v for k, v in globals().get("TRANSECT_STATE", {}).items()
                  if k in _labels}
for lab in _labels:
    TRANSECT_STATE.setdefault(lab, {"on": True, "name": lab, "mids": []})
    TRANSECT_STATE[lab].setdefault("mids", [])
_pos = apply_positions()
_known = [lab for lab in _labels if _pos.get(lab, (None, None))[0] is not None]
_default = (sorted(_known, key=lambda s: -_pos[s][0])          # north to south
            + [lab for lab in _labels if lab not in _known])
TRANSECT_ORDER = [lab for lab in globals().get("TRANSECT_ORDER", []) if lab in _labels]
TRANSECT_ORDER += [lab for lab in _default if lab not in TRANSECT_ORDER]


def _num_or_none(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def _transect_update(order, on, names, mids=None):
    TRANSECT_ORDER[:] = [lab for lab in order if lab in _labels]
    for lab in TRANSECT_ORDER:
        if mids is None:                      # an older list that sends no points
            pts = TRANSECT_STATE.get(lab, {}).get("mids", [])
        else:
            # Half-typed points are kept, box and all, so they survive a
            # re-run; they are only left out when the section is drawn.
            pts = [[_num_or_none(e[0]), _num_or_none(e[1]), (e[2] if len(e) > 2 else None)]
                   for e in (mids.get(lab) or []) if isinstance(e, (list, tuple)) and len(e) >= 2]
        TRANSECT_STATE[lab] = {"on": bool(on.get(lab, True)),
                               "name": (names.get(lab) or "").strip() or lab,
                               "mids": pts}


try:
    from google.colab import output as _colab_output
    _colab_output.register_callback("ctd.transect_update", _transect_update)
except ImportError:
    print("Not in Colab: the list below cannot send changes back, so the default order is used.")


def _transect_rows():
    rows = []
    for lab in TRANSECT_ORDER:
        la, lo = _pos.get(lab, (None, None))
        rows.append({"label": lab, "on": TRANSECT_STATE[lab]["on"],
                     "name": TRANSECT_STATE[lab]["name"], "has_pos": la is not None,
                     "mids": [{"d": m[0], "z": m[1], "to": (m[2] if len(m) > 2 else None)}
                              for m in TRANSECT_STATE[lab].get("mids", [])],
                     "pos": f"{la:.4f}, {lo:.4f}" if la is not None else "no position yet",
                     "color": STATION_COLORS.get(lab, "#333")})
    return json.dumps(rows)


TRANSECT_HTML = r"""<div id="ctd-transect">
<style>
#ctd-transect{font:13px system-ui,-apple-system,'Segoe UI',sans-serif;max-width:720px}
#ctd-transect .hdr{display:flex;gap:8px;color:#777;font-size:11px;padding:0 6px 4px 34px}
#ctd-transect .item{margin:3px 0;border:1px solid #d8d8d8;border-radius:4px;background:#fff;cursor:grab}
#ctd-transect .row{display:flex;align-items:center;gap:8px;padding:4px 6px}
#ctd-transect .item.dragging{opacity:.35}
#ctd-transect .item.over{box-shadow:0 -3px 0 0 #1f77b4}
#ctd-transect .item.below{box-shadow:0 3px 0 0 #1f77b4}
#ctd-transect .grip{color:#aaa;width:14px;text-align:center;user-select:none}
#ctd-transect .n{width:18px;color:#777;font-size:12px;text-align:right}
#ctd-transect .dot{width:10px;height:10px;border-radius:50%;flex:none}
#ctd-transect .lab{width:130px}
#ctd-transect input[type=text]{width:170px;font:13px system-ui,sans-serif;padding:2px 4px;border:1px solid #ccc;border-radius:3px}
#ctd-transect input[type=number]{width:64px;font:12px system-ui,sans-serif;padding:1px 3px;border:1px solid #ccc;border-radius:3px}
#ctd-transect .pos{color:#777;font-size:11px;font-family:monospace}
#ctd-transect .adds{margin-left:auto;display:flex;gap:4px}
#ctd-transect .add{background:none;border:0;font:inherit;color:#1f77b4;font-size:11px;cursor:pointer;white-space:nowrap;padding:0 4px}
#ctd-transect .mids{padding:0 6px 4px 58px;display:flex;flex-direction:column;gap:3px}
#ctd-transect .mid{display:flex;align-items:center;gap:6px;color:#666;font-size:12px}
#ctd-transect .mid .x{background:none;border:0;font:inherit;color:#b00;cursor:pointer;padding:0 4px;font-size:14px}
#ctd-transect .item.off .row,#ctd-transect .item.off .mids{opacity:.55}
</style>
<div class="hdr"><span style="width:18px"></span><span style="width:10px"></span><span style="width:130px">station</span><span style="width:170px">label on the graph</span><span>position</span></div>
<div id="ctd-list"></div>
</div>
<script>
(function () {
  var rows = __ROWS__;
  var list = document.getElementById('ctd-list');
  var from = null;
  function push() {
    var order = rows.map(function (r) { return r.label; });
    var on = {}, names = {}, mids = {};
    rows.forEach(function (r) {
      on[r.label] = r.on; names[r.label] = r.name;
      mids[r.label] = r.mids.map(function (m) { return [m.d, m.z, m.to]; });
    });
    try {
      google.colab.kernel.invokeFunction('ctd.transect_update', [order, on, names, mids], {});
    } catch (e) { console.log('transect update not sent', e); }
  }
  function num(v) { var x = parseFloat(v); return isNaN(x) ? null : x; }
  function live(r) { return r.on && r.has_pos; }
  function nextLive(i) {
    for (var k = i + 1; k < rows.length; k++) { if (live(rows[k])) { return rows[k].label; } }
    return null;
  }
  // where a point was measured towards: a neighbour, or out past an end
  function whither(m) {
    if (m.to === '<before>') { return 'before the line'; }
    if (m.to === '<after>') { return 'beyond the line'; }
    return 'towards ' + (m.to || 'the next station');
  }
  var BETWEEN = 'A seafloor depth in metres known between this station and the next, read off a chart. Shapes the seafloor; the colour between stations stretches down to meet it.';
  var PAST = 'A seafloor depth in metres out past this station, extending the section that way.';
  function noDragWhileTyping(el, inp) {
    inp.addEventListener('focus', function () { el.draggable = false; });
    inp.addEventListener('blur', function () { el.draggable = true; });
  }
  function render() {
    list.innerHTML = '';
    var liveAt = [];
    rows.forEach(function (r, i) { if (live(r)) { liveAt.push(i); } });
    rows.forEach(function (r, i) {
      var el = document.createElement('div');
      el.className = 'item' + (r.on ? '' : ' off');
      el.draggable = true;
      el.innerHTML = '<div class="row"><span class="grip">&#8942;</span><span class="n">' + (i + 1) + '</span>'
        + '<input type="checkbox"' + (r.on ? ' checked' : '') + ' title="on this transect">'
        + '<span class="dot" style="background:' + r.color + '"></span>'
        + '<span class="lab"></span><input type="text">'
        + '<span class="pos"></span><span class="adds"></span></div>'
        + '<div class="mids"></div>';
      el.querySelector('.lab').textContent = r.label;
      el.querySelector('.pos').textContent = r.pos;
      var txt = el.querySelector('input[type=text]');
      txt.value = r.name; txt.placeholder = r.label;
      noDragWhileTyping(el, txt);
      txt.addEventListener('change', function () { r.name = txt.value; push(); });
      el.querySelector('input[type=checkbox]').addEventListener('change', function (e) {
        r.on = e.target.checked; push(); render();
      });
      // a point goes between this station and the next live one, or out past
      // the first or the last, which extends the section that way. A row that
      // is off, unpositioned or alone on the line gets no button.
      var adds = el.querySelector('.adds');
      function addButton(text, to, title) {
        var b = document.createElement('button');
        b.type = 'button'; b.className = 'add'; b.textContent = text; b.title = title;
        b.addEventListener('click', function () { r.mids.push({ d: null, z: null, to: to }); push(); render(); });
        adds.appendChild(b);
      }
      if (live(r) && liveAt.length > 1) {
        if (i === liveAt[0]) { addButton('+ point before', '<before>', PAST); }
        if (nextLive(i) !== null) { addButton('+ point after', nextLive(i), BETWEEN); }
        if (i === liveAt[liveAt.length - 1]) { addButton('+ point after', '<after>', PAST); }
      }
      var midsEl = el.querySelector('.mids');
      r.mids.forEach(function (m, j) {
        var line = document.createElement('div');
        line.className = 'mid';
        line.innerHTML = '<span>&#8627;</span><span class="to"></span><input type="number" step="0.01" min="0" placeholder="km"> km, '
          + '<input type="number" step="0.1" min="0" placeholder="m"> m deep<button type="button" class="x" title="remove">&#215;</button>';
        line.querySelector('.to').textContent = whither(m);
        var ins = line.querySelectorAll('input');
        if (m.d !== null && m.d !== undefined) { ins[0].value = m.d; }
        if (m.z !== null && m.z !== undefined) { ins[1].value = m.z; }
        ins.forEach(function (inp) { noDragWhileTyping(el, inp); });
        ins[0].addEventListener('change', function () { m.d = num(ins[0].value); push(); });
        ins[1].addEventListener('change', function () { m.z = num(ins[1].value); push(); });
        line.querySelector('.x').addEventListener('click', function () { r.mids.splice(j, 1); render(); push(); });
        midsEl.appendChild(line);
      });
      // drag the whole item; its points travel with it, still anchored to
      // the neighbour they were measured towards
      el.addEventListener('dragstart', function (e) { from = i; el.classList.add('dragging'); e.dataTransfer.effectAllowed = 'move'; try { e.dataTransfer.setData('text/plain', String(i)); } catch (x) {} });
      el.addEventListener('dragend', function () { from = null; render(); });
      el.addEventListener('dragover', function (e) { e.preventDefault(); e.dataTransfer.dropEffect = 'move'; el.classList.add(from !== null && from < i ? 'below' : 'over'); });
      el.addEventListener('dragleave', function () { el.classList.remove('over', 'below'); });
      el.addEventListener('drop', function (e) {
        e.preventDefault();
        if (from === null || from === i) { render(); return; }
        var moved = rows.splice(from, 1)[0];
        rows.splice(i, 0, moved);
        from = null; render(); push();
      });
      list.appendChild(el);
    });
  }
  render();
})();
</script>"""

display(widgets.HTML(
    "<b>Stations on this transect</b> &mdash; untick any that are not on the line, "
    "and drag the rows into the order they lie along it. Order is geographic: how "
    "the stations sit relative to one another across the water, not what they are "
    "called. The starting guess is north to south. The text box is the label that "
    "appears above the station on the graph.<br>"
    "<b>+ point after</b> adds a depth you know between that station and the next, "
    "read off a chart in metres (convert feet or fathoms first): how far along the line "
    "from the station, and how deep. <b>+ point before</b> on the first station, or "
    "<b>+ point after</b> on the last, puts a point out past that end and extends the "
    "section that way, coloured like the end station. Points shape the black seafloor on the graph; the "
    "colour between stations is stretched down to meet it, so anything below a cast's "
    "deepest reading is inferred, not measured. A point stays with its station and the "
    "neighbour it was measured towards if you drag the rows about. Known bathymetry "
    "pulled from online is planned for later.<br>"
    "Positions typed in above are used as they are; re-run this cell to see them "
    "in the list."))
display(HTML(TRANSECT_HTML.replace("__ROWS__", _transect_rows())))
print("\nNow run cell 4 for the map and the transect.")


In [ ]:
#@title 4 · Station map and water column transect
#@markdown A vertical section (transect plot): distance along the line of stations against depth, coloured by one variable and interpolated between the casts.
#@markdown
#@markdown **Contour detail** — 0 draws a smooth gradient; any other number bands the colour range into that many steps.
#@markdown Colours are fixed per variable (`VARIABLE_RANGES` in Setup), so the same colour means the same value on every section.
contour_steps = 0 #@param {type:"slider", min:0, max:50, step:1}
#@markdown **Your own palette** — upload the file through the Files sidebar (folder icon on the left)
#@markdown and put its path here, such as `/content/my.cpt`; blank means none. Surfer .clr and .lvl,
#@markdown GMT .cpt, ODV .pal, Ferret .spk, NCL .rgb, SNAP .cpd, GIMP .ggr, ParaView .json and .xml,
#@markdown QGIS ramps, GRASS and GDAL rules. A palette with levels (real values) sets the colour range too.
palette_file = "" #@param {type:"string"}
#@markdown Which variable it colours: blank means the first ticked one. `all` colours every section,
#@markdown but only with a palette that has no levels, since levels belong to one variable.
palette_for = "" #@param {type:"string"}
colour_bar_label = "units" #@param ["units", "name and units"]

try:
    TRANSECT_ORDER
except NameError:
    raise SystemExit("Run cell 3 first (positions and transect).")

LOCATION = LOCATION_BOX.value.strip()
_pos = apply_positions()
order = [lab for lab in TRANSECT_ORDER
         if TRANSECT_STATE[lab]["on"] and None not in _pos.get(lab, (None, None))]
names = {lab: TRANSECT_STATE[lab]["name"] for lab in order}
# every station's points go in; seafloor_points() says which are not used
mids = {lab: TRANSECT_STATE[lab].get("mids", []) for lab in TRANSECT_ORDER
        if TRANSECT_STATE[lab].get("mids")}
_skipped = [lab for lab in TRANSECT_ORDER
            if TRANSECT_STATE[lab]["on"] and lab not in order]
if _skipped:
    print("No position yet, left off the transect:", ", ".join(_skipped))

# Say which seafloor points are not used, and why, and how far a point past
# either end extends the section, before drawing anything.
_have, _dists = transect_distances(order, _pos)
if len(_have) >= 2 and mids:
    _bottoms = [0.0] * len(_have)
    _pts, _notes, _used, _xmin, _xmax = seafloor_points(_have, _dists, _bottoms, mids)
    for _n in _notes:
        print(_n)
    if _used:
        print(f"{_used} seafloor point(s) used")

series = selected_series()
if not series:
    raise SystemExit("Nothing ticked to plot in the Load cell.")

# Your own palette: it colours one section, or every section for a palette
# without levels. Levels are real values, so a palette that has them sets the
# colour range and belongs to one variable.
palette, palette_all, _pal_for = None, False, palette_for.strip()
if palette_file.strip():
    try:
        with open(palette_file.strip(), encoding="utf-8-sig", errors="replace") as _fh:
            palette = read_palette(_fh.read(), palette_file.strip())
        print(f"Palette {os.path.basename(palette_file.strip())}: {palette['note']}")
    except Exception as e:
        print(f"Palette not used: {e}")
if palette:
    if _pal_for.lower() == "all" and palette["levels"]:
        print("  'all' needs a palette without levels; this one goes on the first ticked variable")
        _pal_for = ""
    palette_all = _pal_for.lower() == "all"
    if not palette_all:
        hit = [s["name"] for s in series if s["name"].lower() == _pal_for.lower()]
        if not _pal_for:
            _pal_for = series[0]["name"]
        elif not hit:
            print(f"  nothing ticked is called '{_pal_for}'; ticked: "
                  + ", ".join(s["name"] for s in series))
            palette = None
        else:
            _pal_for = hit[0]
    if palette:
        print("  colours " + ("every section" if palette_all else f"the {_pal_for} section"))


def _num(s):
    s = str(s).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


dmin = _num(globals().get("depth_from_m", ""))
dmax = _num(globals().get("depth_to_m", ""))
if dmin is not None and dmax is not None and dmin > dmax:
    dmin, dmax = dmax, dmin

figs = []
mp = station_map()
if mp is not None:
    figs.append(("Station map", mp))
    print(f"Station map: {sum(1 for m in STATION_META.values() if m.get('lat') is not None)}"
          f" of {len(stations)} stations positioned")
else:
    print("Station map: skipped, no positions entered")

made = []
if len(order) < 2:
    print("Transect: needs at least two ticked stations with positions.")
else:
    for s in series:
        use = palette if palette and (palette_all or s["name"] == _pal_for) else None
        sec = build_section(stations, s, _pos, order=order, depth_min=dmin,
                            depth_max=dmax, n_contours=contour_steps, names=names,
                            midpoints=mids,
                            colorscale_stops=use["stops"] if use else None,
                            color_range=((use["levels"][0], use["levels"][-1])
                                         if use and use["levels"] else None),
                            colorbar_name=(colour_bar_label == "name and units"))
        if sec is not None:
            figs.append((f"{s['name']} section", sec))
            made.append(s["name"])
    if made:
        _, d = transect_distances(order, _pos)
        detail = "smooth gradient" if not contour_steps else f"{contour_steps} contour steps"
        print(" → ".join(names[lab] for lab in order))
        print(f"{d[-1]:.2f} km end to end · {len(made)} sections · {detail}")
    else:
        print("Transect: nothing to draw. Check the ticked variables and the depth window.")

if figs:
    print("Saved:", export_figures(figs, tag=" · transect", stem="CTD_transect"))
    # show_inline, not f.show(): after the widgets in cell 3, Colab can run the
    # plot script before plotly.js has loaded and leave an empty box. See Setup.
    for _n, f in figs:
        show_inline(f)
